# Capítulo 12: Árvores de Decisão

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde à seção 8.1 de James et al. (2023).

> Embora as árvores tirem A+ em interpretabilidade, são preditores bons, mas não ótimos. Deem a elas, digamos, um B em previsão.
>
> — Leo Breiman, em tradução livre (2001, p. 207)

Uma árvore de regressão prevê com perguntas de sim ou não. Nos jogadores de beisebol de `Hitters`, com o logaritmo do salário como resposta, cada pergunta corta o espaço dos preditores em dois, as folhas são retângulos, e a previsão para quem cai numa folha é a média da resposta dos jogadores de treino que caíram nela. A seção 12.1 mostra como os cortes são escolhidos: pela divisão binária recursiva, que começa na raiz e, a cada passo, fica com o corte que mais reduz o RSS naquele momento, sem olhar se um corte pior agora abriria caminho para cortes melhores depois. Deixada crescer, a árvore decora o treino. A seção 12.2 cresce uma árvore grande e a poda por custo-complexidade, com uma penalidade $\alpha$ por folha, parecida com a do lasso, e escolhe $\alpha$ por validação cruzada; nos jogadores, fica uma árvore de 4 folhas que erra menos no teste que a árvore grande. Com outro sorteio dos jogadores de treino, porém, o tamanho escolhido muda bastante.

Quando a resposta é uma classe, como o resultado de uma angiografia nos pacientes de `Heart`, a folha prevê a classe mais comum entre os seus pacientes de treino. A seção 12.3 mostra por que a taxa de erro de classificação é um critério ruim para escolher os cortes: um corte em que os dois filhos continuam prevendo a mesma classe que o pai não muda o erro, por mais que um dos filhos fique quase puro. O índice de Gini e a entropia, as duas medidas de impureza que o substituem, registram esse ganho. A poda segue o caminho da seção 12.2, agora com a validação cruzada medindo o erro de classificação, e a árvore escolhida usa três das 13 medidas dos pacientes.

A regressão linear supõe que cada preditor soma um efeito que não depende dos outros; a árvore supõe uma resposta constante dentro de cada caixa. A seção 12.4 põe as duas formas lado a lado: num simulado com fronteira diagonal, a regressão logística ganha, e num com fronteira em L, ganha a árvore. Em dados reais, quem decide é o erro estimado por validação cruzada aninhada, com a escolha do tamanho da árvore refeita dentro de cada grupo: a árvore ganha nos jogadores, e a logística, nos pacientes. A seção 12.5 pesa o que sobra. A favor da árvore, uma árvore pequena se lê sem fórmula nenhuma, e o método pode dividir uma coluna de texto em dois grupos de categorias; com colunas indicadoras, que é como o `scikit-learn` as recebe, cada corte separa só uma categoria das demais. Contra ela, a precisão e a instabilidade: duas metades sorteadas dos mesmos jogadores dão árvores que começam por colunas diferentes, e a previsão da árvore para um mesmo jogador varia mais de uma amostra para outra que a da regressão linear.

Ao final deste capítulo, você será capaz de:

- Descrever uma árvore de regressão como uma divisão do espaço dos preditores em retângulos, explicar a divisão binária recursiva e ajustar e desenhar uma árvore com `DecisionTreeRegressor` e `plot_tree`
- Podar uma árvore por custo-complexidade com `cost_complexity_pruning_path`, converter o $\alpha$ da penalidade no `ccp_alpha` do `scikit-learn` e escolher o tamanho da árvore com `GridSearchCV`
- Calcular a taxa de erro de classificação, o índice de Gini e a entropia de um nó, e explicar por que as duas últimas são as usadas para crescer a árvore e a primeira para julgar a árvore podada
- Comparar uma árvore com um modelo linear pela forma da fronteira ou da função que cada um supõe e pelo erro estimado por validação cruzada aninhada
- Avaliar quando uma árvore é uma boa escolha, pesando a leitura fácil e o tratamento de colunas de texto contra a precisão e a instabilidade diante de outra amostra de treino

## Seções

| Seção | Tópico |
|---|---|
| [12.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/01-arvores-de-regressao.html) | Árvores de Regressão |
| [12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-podando-a-arvore.html) | Podando a Árvore |
| [12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-arvores-de-classificacao.html) | Árvores de Classificação |
| [12.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/04-arvores-contra-modelos-lineares.html) | Árvores contra Modelos Lineares |
| [12.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/05-vantagens-e-desvantagens.html) | Vantagens e Desvantagens das Árvores |

## Árvores de Regressão

> **📌 Nota**
>
> Esta seção corresponde à seção 8.1.1 de James et al. (2023).

O salário de um jogador de beisebol tende a crescer com os anos que ele tem nas grandes ligas e com o desempenho na temporada anterior, mas não precisa crescer em linha reta, nem do mesmo jeito para um novato e para um veterano. Dá para prever o salário de um jogador com duas ou três perguntas de sim ou não sobre ele?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor, plot_tree

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### Os salários

O conjunto `Hitters` reúne dados de jogadores das grandes ligas de beisebol norte-americanas: as estatísticas da temporada de 1986 e as mesmas estatísticas acumuladas na carreira (as colunas terminadas em `_carreira`), além do salário na temporada de 1987, em milhares de dólares. Os preditores são só duas colunas: `anos`, o número de anos do jogador nas grandes ligas, e `rebatidas`, o número de rebatidas na temporada anterior.

In [ ]:
hitters = pd.read_csv("dados/Hitters.csv")
print("jogadores:", len(hitters))
print("sem salário:", hitters["salario"].isna().sum())
hitters = hitters.dropna()
print("depois do dropna:", len(hitters))
print("assimetria do salário:", numero_br(hitters["salario"].skew()))
y = np.log(hitters["salario"])
print("assimetria do log:", numero_br(y.skew()))

> **🔧 Função**
>
> **`pd.read_csv(caminho)`** — lê um arquivo CSV e devolve um `DataFrame`, uma coluna por campo do cabeçalho.
>
> **`s.isna().sum()`** — `isna` marca com `True` cada valor ausente da série; a soma conta os `True`.
>
> **`df.dropna()`** — devolve a tabela sem as linhas que têm algum valor ausente.
>
> **`s.skew()`** — a assimetria da série: perto de zero numa distribuição simétrica, positiva quando há uma cauda longa à direita.
>
> **`np.log(x)`** — o logaritmo natural de cada elemento.

Dos 322 jogadores, 59 não têm salário registrado, e sem a resposta não há o que prever: ficam 263. O salário tem assimetria de 1,59: uns poucos jogadores muito bem pagos formam uma cauda longa à direita. Como a árvore vai prever a média de cada grupo de jogadores, um grupo com um desses salários enormes teria a média puxada para cima por ele. O logaritmo do salário tem assimetria de −0,18, quase simétrico, e nele os salários enormes deixam de dominar a média. A resposta daqui em diante é `y`, o logaritmo do salário; a exponencial, `np.exp`, leva de volta à escala do salário, em milhares de dólares.

### Uma árvore com três folhas

Uma árvore de regressão faz perguntas de sim ou não sobre os preditores, uma de cada vez, e prevê para cada jogador a média da resposta dos jogadores de treino que seguiram o mesmo caminho de perguntas. A árvore é feita de **nós**: cada pergunta é um nó, e cada saída, sim ou não, é um **ramo** que sai dele e leva a outro nó. Os nós do fim, em que não se pergunta mais nada, são as **folhas**, e cada folha é um grupo final de jogadores. O `DecisionTreeRegressor` do `scikit-learn` constrói a árvore; com `max_leaf_nodes=3`, ela para quando tiver três folhas.

In [ ]:
X2 = hitters[["anos", "rebatidas"]]
arvore = DecisionTreeRegressor(max_leaf_nodes=3, random_state=12)
arvore.fit(X2, y)
print("folhas:", arvore.get_n_leaves())
print("profundidade:", arvore.get_depth())

> **🔧 Função**
>
> **`DecisionTreeRegressor(max_leaf_nodes, random_state)`** — a árvore de regressão. `max_leaf_nodes` limita o número de folhas: a árvore cresce dividindo, a cada passo, o grupo cuja divisão mais reduz o erro de treino, e para ao chegar ao limite. `random_state` fixa a ordem em que as colunas são examinadas, que decide empates entre dois cortes igualmente bons.
>
> **`arvore.fit(X, y)`** — constrói a árvore com as linhas de `X` e a resposta `y`.
>
> **`arvore.get_n_leaves()`**, **`arvore.get_depth()`** — o número de folhas e o maior número de perguntas feitas a um jogador até chegar a uma delas.

Três folhas, e no máximo duas perguntas por jogador. Quais perguntas, a figura mostra. O `plot_tree` desenha a árvore em inglês, com o ponto decimal; a função `em_portugues`, abaixo, traduz o texto de cada caixa e dá a ela um fundo branco, para que a seta não atravesse o texto:

In [ ]:
import re

def em_portugues(anotacoes):
    """Traduz e ajusta as caixas que o plot_tree desenha."""
    for anotacao in anotacoes:
        texto = (anotacao.get_text()
                 .replace("samples", "n").replace("value", "média")
                 .replace("<=", "≤")
                 .replace("True", "sim").replace("False", "não"))
        texto = re.sub(r"(\d)\.(\d)", r"\1,\2", texto)
        anotacao.set_text(texto)
        caixa = anotacao.get_bbox_patch()
        if caixa is not None:
            caixa.set(facecolor="white", edgecolor="0.45")

> **🔧 Função**
>
> **`em_portugues(caixas)`** — para cada caixa, lê o texto (`get_text`), troca os termos em inglês, põe vírgula no lugar do ponto entre dois dígitos (`re.sub`, que substitui cada trecho que casa com um padrão) e grava o texto novo (`set_text`); depois pinta de branco a caixa em volta dele (`get_bbox_patch().set`).

In [ ]:
# Figura: A árvore de três folhas para o logaritmo do salário. Cada caixa mostra a pergunta, o número de jogadores (n) e a média do logaritmo do salário deles. Em cada pergunta, o ramo "sim" é o da esquerda, em que a condição vale.
fig, ax = plt.subplots(figsize=(4.0, 3.0))
caixas = plot_tree(arvore, feature_names=list(X2.columns), impurity=False,
                   rounded=True, precision=3, fontsize=10, ax=ax)
em_portugues(caixas)
plt.show()

> **🔧 Função**
>
> **`plt.subplots(figsize=(largura, altura))`** — cria uma figura, com o tamanho em polegadas, e o eixo `ax` em que se desenha. **`plt.show()`** — exibe a figura pronta.
>
> **`plot_tree(arvore, feature_names, impurity, rounded, precision, fontsize, ax)`** — desenha a árvore ajustada e devolve a lista das caixas de texto. `feature_names` dá os nomes das colunas (sem ele, sai `x[0]`); `impurity=False` omite o erro de cada nó; `precision` é o número de casas decimais; `rounded=True` arredonda os cantos das caixas; `fontsize` é o tamanho da letra.

A primeira pergunta é sobre `anos`. Os jogadores com até 4 anos de liga vão para uma folha só, e nenhuma pergunta sobre rebatidas é feita a eles. Os que têm 5 anos ou mais são separados de novo pelas rebatidas da temporada anterior, em 117,5. A tabela abaixo resume as três folhas. A coluna "típico" é a exponencial da média do logaritmo, em dólares; a coluna "média" é a média dos salários da folha, também em dólares:

In [ ]:
folhas = pd.DataFrame({"folha": arvore.apply(X2), "log": y,
                       "dolares": hitters["salario"] * 1000})
resumo = folhas.groupby("folha").agg(n=("log", "size"),
                                     log=("log", "mean"),
                                     media=("dolares", "mean"))
resumo.index = ["R1", "R2", "R3"]
resumo.insert(2, "tipico", np.exp(resumo["log"]) * 1000)
print("típico abaixo da média:",
      bool((resumo["tipico"] < resumo["media"]).all()))
resumo.columns = ["n", "média do log", "típico", "média"]
resumo.style.format({"média do log": lambda v: numero_br(v, 3),
                     "típico": lambda v: numero_br(v, 0),
                     "média": lambda v: numero_br(v, 0)})

> **🔧 Função**
>
> **`pd.DataFrame({nome: coluna, ...})`** — monta uma tabela a partir de um dicionário, uma coluna por chave. **`arvore.apply(X)`** — o número do nó em que cada linha termina, isto é, a folha de cada jogador.
>
> **`df.groupby("folha").agg(nome=(coluna, funcao), ...)`** — separa as linhas pela folha e calcula, em cada grupo, uma coluna nova por argumento: aqui, o número de linhas (`"size"`) e as médias (`"mean"`) do logaritmo e do salário.
>
> **`np.exp(x)`** — a exponencial de cada elemento, a inversa do `np.log`. **`df.insert(posicao, nome, valores)`** — acrescenta uma coluna na posição dada.
>
> **`df.style.format({coluna: funcao})`** — exibe a tabela passando cada valor da coluna pela função; aqui, a `numero_br`.

Os novatos ($R_1$, 90 jogadores) têm média 5,107 no logaritmo, isto é, um salário típico de US$ 165.139. Os veteranos com até 117 rebatidas ($R_2$, 90) ficam em US$ 402.776, e os que rebateram mais ($R_3$, 83), em US$ 845.296. Nas três folhas o salário típico fica abaixo da média dos salários: a exponencial da média do logaritmo é a média geométrica, que os poucos salários enormes puxam menos. Nesta árvore de três folhas, os anos de liga separam primeiro; o desempenho recente só entra entre os veteranos, porque o limite de folhas acaba antes de uma pergunta sobre rebatidas chegar aos novatos. É uma simplificação, porque três médias não contam a história inteira de 263 salários, mas é uma que se explica a um dirigente de clube sem uma fórmula.

### Regiões do espaço dos preditores

Cada jogador é um ponto no plano de `anos` e `rebatidas`, e as duas perguntas da árvore cortam esse plano em três retângulos. Os pontos de corte estão guardados na árvore ajustada, e a última linha confere se as duas colunas só têm valores inteiros:

In [ ]:
internos = arvore.tree_.feature >= 0
colunas = X2.columns[arvore.tree_.feature[internos]]
limiares = arvore.tree_.threshold[internos]
for coluna, limiar in zip(colunas, limiares):
    print(f"{coluna} ≤ {numero_br(limiar, 1)}")
corte_anos, corte_rebatidas = limiares
print("valores inteiros:", bool((X2 % 1 == 0).all().all()))

> **🔧 Função**
>
> **`arvore.tree_.feature`** — para cada nó, o índice da coluna usada na pergunta; nas folhas, que não perguntam nada, vale −2. **`arvore.tree_.threshold`** — o limiar da pergunta de cada nó. Os nós são numerados na ordem em que a árvore os cria, a partir da raiz, que é o nó 0.
>
> **`(X2 % 1 == 0).all().all()`** — o primeiro `.all()` dá um booleano por coluna (todos os valores inteiros?); o segundo resume as colunas num só.

As três regiões são

$$
\begin{aligned}
R_1 &= \{X \mid \text{anos} < 4{,}5\},\\
R_2 &= \{X \mid \text{anos} \geq 4{,}5,\\
    &\qquad\ \text{rebatidas} < 117{,}5\},\\
R_3 &= \{X \mid \text{anos} \geq 4{,}5,\\
    &\qquad\ \text{rebatidas} \geq 117{,}5\}.
\end{aligned}
$$

Como `anos` e `rebatidas` são números inteiros, nenhum jogador fica em cima de um corte, e "< 4,5" e "≤ 4,5" separam os mesmos jogadores: quem tem até 4 anos de quem tem 5 ou mais.

In [ ]:
# Figura: Os 263 jogadores no plano de anos de liga e rebatidas, com a cor da região em que a árvore de três folhas põe cada um: $R_1$ azul, $R_2$ verde, $R_3$ roxo. A linha vertical é o corte em 4,5 anos; a horizontal, o corte em 117,5 rebatidas, que só existe à direita dela.
cor = {1: "C0", 3: "C2", 4: "C3"}
fig, ax = plt.subplots(figsize=(4.8, 3.6))
ax.scatter(X2["anos"], X2["rebatidas"], s=12, alpha=0.8,
           c=[cor[f] for f in arvore.apply(X2)])
ax.axvline(corte_anos, color="0.45", linewidth=1.5)
ax.hlines(corte_rebatidas, corte_anos, 25, color="0.45", linewidth=1.5)
for rotulo, (x, y_texto) in {"$R_1$": (2.5, 252), "$R_2$": (21, 20),
                             "$R_3$": (21, 225)}.items():
    ax.text(x, y_texto, rotulo, fontsize=12, ha="center", va="center")
ax.set_xlim(0, 25)
ax.set_ylim(0, 265)
ax.set_xlabel("anos de liga")
ax.set_ylabel("rebatidas em 1986")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.scatter(x, y, s, alpha, c)`** — um ponto por par `(x, y)`; `s` é o tamanho, `alpha` a opacidade e `c` a cor de cada ponto.
>
> **`ax.axvline(x, color, linewidth)`** — uma linha vertical na posição `x`, de alto a baixo do gráfico. **`ax.hlines(y, xmin, xmax, color, linewidth)`** — uma linha horizontal na altura `y`, só de `xmin` a `xmax`.
>
> **`ax.text(x, y, texto, fontsize, ha, va)`** — escreve `texto` na posição `(x, y)`; `ha` e `va` alinham o texto pelo centro.
>
> **`ax.set_xlim`**, **`ax.set_ylim`** — os limites dos eixos; **`ax.set_xlabel`**, **`ax.set_ylabel`** — os nomes dos eixos. **`plt.tight_layout()`** — ajusta as margens para que nada fique cortado.

A árvore e o plano são duas figuras do mesmo modelo: as regiões $R_1$, $R_2$ e $R_3$ são as folhas, também chamadas **nós terminais**, e as perguntas no caminho até elas são os **nós internos**. A árvore é desenhada de cabeça para baixo, com a raiz no topo e as folhas embaixo.

> **🔷 Conceito**
>
> Uma **árvore de regressão** divide o espaço dos preditores em $J$ regiões retangulares $R_1, \dots, R_J$, que não se sobrepõem, e prevê para toda observação que cai em $R_j$ o mesmo valor $\hat y_{R_j}$: a média da resposta das observações de treino de $R_j$.

Um jogador com 6 anos de liga e 130 rebatidas cai em $R_3$. Qual salário a árvore prevê para ele? E para um com 6 anos e 60 rebatidas?

### Como a árvore escolhe o corte

As regiões podiam ter qualquer forma; a árvore as restringe a retângulos, porque é isso que permite descrevê-las por perguntas simples. Entre todas as divisões em $J$ retângulos, a melhor seria a que minimiza o RSS

$$
\sum_{j=1}^{J} \sum_{i \in R_j} \left(y_i - \hat y_{R_j}\right)^2. \tag{8.1}
$$

Com três retângulos e duas colunas, ainda daria para testar todas as divisões; com dezenas de retângulos, o número de divisões possíveis cresce tanto que a busca completa deixa de ser viável. A árvore faz uma busca mais modesta, a **divisão binária recursiva**: começa com todos os jogadores numa região só e escolhe **um** corte, o melhor naquele momento; depois repete dentro de cada pedaço. É uma busca de cima para baixo, porque começa na raiz, e **gulosa**, porque cada corte é o melhor para aquele passo, sem olhar se um corte pior agora abriria caminho para cortes melhores depois.

Para uma coluna $X_j$ e um ponto de corte $s$, os dois semiplanos são

$$
\begin{aligned}
R_1(j, s) &= \{X \mid X_j < s\},\\
R_2(j, s) &= \{X \mid X_j \geq s\}.
\end{aligned}
\tag{8.2}
$$

Com $X_j$ = `anos` e $s = 4{,}5$, $R_1(j, s)$ são os 90 novatos da folha $R_1$, e $R_2(j, s)$ são os veteranos, as folhas $R_2$ e $R_3$ juntas. O corte escolhido é o par $(j, s)$ que minimiza

$$
\begin{aligned}
&\sum_{i:\, x_i \in R_1(j, s)} \left(y_i - \hat y_{R_1}\right)^2\\
&\quad + \sum_{i:\, x_i \in R_2(j, s)} \left(y_i - \hat y_{R_2}\right)^2,
\end{aligned}
\tag{8.3}
$$

em que $\hat y_{R_1}$ e $\hat y_{R_2}$ são as médias da resposta em cada lado. No corte em 4,5 anos, $\hat y_{R_1}$ é 5,107, a média do logaritmo dos novatos, e (8.3) soma os quadrados das distâncias de cada novato a ela com os de cada veterano à média dos veteranos. Com `anos` e `rebatidas`, os pontos de corte que importam são os pontos médios entre dois valores vizinhos da coluna: qualquer outro $s$ entre o menor e o maior valor separa os mesmos jogadores que um deles.

Antes de fazer a conta: olhando a figura das regiões, em qual das duas colunas você cortaria primeiro?

Na prática, quem faz essa busca é o `fit` do `DecisionTreeRegressor`. A função abaixo só mede o que ele compara: calcula (8.3) com `pandas` para cada ponto de corte de uma coluna, e o chunk logo depois da conta confere que ela chega ao mesmo corte que a árvore ajustada.

In [ ]:
def rss_do_corte(X, y, coluna):
    """O RSS (8.3) de cada ponto de corte da coluna."""
    valores = np.sort(X[coluna].unique())
    cortes = (valores[:-1] + valores[1:]) / 2
    rss = []
    for s in cortes:
        esq = X[coluna] < s
        rss.append(((y[esq] - y[esq].mean()) ** 2).sum()
                   + ((y[~esq] - y[~esq].mean()) ** 2).sum())
    return pd.Series(rss, index=cortes)

tss = ((y - y.mean()) ** 2).sum()
print("RSS sem corte:", numero_br(tss))
curvas = {c: rss_do_corte(X2, y, c) for c in X2.columns}
for coluna, curva in curvas.items():
    print(f"{coluna}: {len(curva)} cortes")
    print(f"  menor RSS {numero_br(curva.min())}"
          f" em {numero_br(curva.idxmin(), 1)}")

> **🔧 Função**
>
> **`np.sort(s.unique())`** — os valores distintos da coluna, em ordem crescente.
>
> **`curva.idxmin()`** — o rótulo do menor valor da série; aqui, o ponto de corte de menor RSS.
>
> **`pd.Series(valores, index=rotulos)`** — uma coluna solta, com um rótulo por valor; `rss_do_corte` a devolve com os pontos de corte como rótulos. **`s.mean()`** e **`s.min()`** — a média e o menor valor.

Sem nenhum corte, o RSS é 207,15, a soma dos quadrados em torno da média geral. O melhor corte em `anos` leva o RSS a 115,06, em 4,5; o melhor em `rebatidas`, a 160,97, em 117,5. O corte em `anos` reduz mais o RSS e vira a raiz. É o mesmo corte que o `DecisionTreeRegressor` escolheu:

In [ ]:
coluna_raiz = X2.columns[arvore.tree_.feature[0]]
melhor = min(curvas, key=lambda c: curvas[c].min())
print("coluna igual:", melhor == coluna_raiz)
print("limiar igual:",
      curvas[melhor].idxmin() == arvore.tree_.threshold[0])

In [ ]:
# Figura: O RSS (8.3) de cada ponto de corte, em anos de liga (em cima) e em rebatidas (embaixo), no mesmo eixo vertical. A linha tracejada cinza é o RSS sem corte; o x marca o menor RSS de cada coluna.
fig, eixos = plt.subplots(2, 1, figsize=(4.8, 5.0), sharey=True)
rotulos = {"anos": "ponto de corte em anos",
           "rebatidas": "ponto de corte em rebatidas"}
limites = {"anos": (0, 25), "rebatidas": (0, 250)}
for ax, (coluna, curva) in zip(eixos, curvas.items()):
    ax.plot(curva.index, curva.values, color="C0", linewidth=2)
    ax.scatter(curva.idxmin(), curva.min(), marker="x", s=80, color="C0",
               linewidths=2.5, zorder=3)
    ax.axhline(tss, color="0.45", linestyle="--", linewidth=1.5)
    ax.set_xlim(limites[coluna])
    ax.set_xlabel(rotulos[coluna])
    ax.set_ylabel("RSS")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`plt.subplots(2, 1, figsize, sharey=True)`** — dois gráficos empilhados, com o mesmo eixo vertical, para que as alturas se comparem a olho.
>
> **`ax.plot(x, y, color, linewidth)`** — uma linha ligando os pontos `(x, y)` em ordem. **`ax.axhline(y, color, linestyle)`** — uma linha horizontal na altura `y`, de ponta a ponta do gráfico; `linestyle="--"` a faz tracejada.

O passo seguinte repete a conta dentro de cada região. Os novatos e os veteranos são dois grupos separados, e em cada um se procura o melhor corte das duas colunas. O ganho de um corte é quanto ele reduz o RSS do grupo:

In [ ]:
grupos = {"novatos": X2["anos"] < corte_anos,
          "veteranos": X2["anos"] >= corte_anos}
for nome, linhas in grupos.items():
    y_grupo = y[linhas]
    rss_grupo = ((y_grupo - y_grupo.mean()) ** 2).sum()
    ganhos = {}
    for coluna in X2.columns:
        curva = rss_do_corte(X2[linhas], y_grupo, coluna)
        ganhos[coluna] = rss_grupo - curva.min()
        print(f"{nome}, {coluna}:")
        print(f"  ganho {numero_br(ganhos[coluna])}"
              f" em {numero_br(curva.idxmin(), 1)}")
    print(f"{nome}, melhor: {max(ganhos, key=ganhos.get)}")

Entre os veteranos, o melhor corte é em `rebatidas`, em 117,5, com um ganho de 23,73. Entre os novatos, o melhor também seria em `rebatidas`, em 15,5, mas com um ganho de 9,34. Com só mais uma folha permitida, a árvore divide o grupo em que o ganho é maior, o dos veteranos. Os novatos ficam sem a pergunta sobre rebatidas porque o limite de três folhas acabou antes, não porque o corte deixe de reduzir o RSS deles.

### Quando parar

Sem o `max_leaf_nodes`, a árvore continua a dividir até não ter mais o que dividir. O RSS de treino a cada tamanho:

In [ ]:
linhas = []
for limite in [2, 3, 4, 5, 8, 20, 50, None]:
    modelo = DecisionTreeRegressor(max_leaf_nodes=limite, random_state=12)
    modelo.fit(X2, y)
    residuo = y - modelo.predict(X2)
    linhas.append({"folhas": modelo.get_n_leaves(),
                   "profundidade": modelo.get_depth(),
                   "RSS": (residuo ** 2).sum()})
tamanhos = pd.DataFrame(linhas)
print("RSS sempre desce:", tamanhos["RSS"].is_monotonic_decreasing)
tamanhos.style.hide(axis="index").format({"RSS": lambda v: numero_br(v)})

> **🔧 Função**
>
> **`modelo.predict(X)`** — a previsão da árvore para cada linha de `X`: a média da folha em que a linha cai.
>
> **`s.is_monotonic_decreasing`** — `True` se cada valor da série é menor ou igual ao anterior.
>
> **`.style.hide(axis="index")`** — esconde a coluna do índice na tabela exibida.

Com 2 folhas, o RSS de treino é 115,06, o do primeiro corte; com 50, é 21,04. A última linha é a árvore sem limite: 248 folhas para 263 jogadores, com até 18 perguntas entre a raiz e uma folha, e RSS de treino de 0,73. Por que não zero, se há quase uma folha por jogador? Porque há jogadores com os mesmos `anos` e as mesmas `rebatidas` e salários diferentes, e nenhuma pergunta sobre as duas colunas os separa:

In [ ]:
salarios = hitters.groupby(["anos", "rebatidas"])["salario"].nunique()
print("pares com salários diferentes:", (salarios > 1).sum())

> **🔧 Função**
>
> **`s.nunique()`** — o número de valores distintos da série; depois do `groupby`, um por par de `anos` e `rebatidas`.

Em 9 pares de valores de `anos` e `rebatidas`, os jogadores têm salários diferentes, e a folha deles só pode prever a média. Nos tamanhos da tabela, o RSS de treino só desce: julgada por ele, a melhor árvore é sempre a maior.

O que a árvore sem limite prevê para os dois jogadores da pergunta anterior, com 6 anos de liga e 130 ou 60 rebatidas? Compare com a árvore de três folhas:

In [ ]:
cheia = DecisionTreeRegressor(random_state=12).fit(X2, y)
por_folha = pd.Series(cheia.apply(X2)).value_counts()
print("folhas com 1 jogador:", (por_folha == 1).sum(), "de", len(por_folha))
novos = pd.DataFrame({"anos": [6, 6], "rebatidas": [130, 60]})
for nome, modelo in [("3 folhas", arvore), ("sem limite", cheia)]:
    previsto = np.exp(modelo.predict(novos)) * 1000
    print(f"{nome}: US$ {numero_br(previsto[0], 0)}"
          f" e US$ {numero_br(previsto[1], 0)}")
print("jogadores nessas folhas:", por_folha[cheia.apply(novos)].tolist())

> **🔧 Função**
>
> **`s.value_counts()`** — quantas vezes cada valor aparece na série; aqui, quantos jogadores caem em cada folha. Indexada pelas folhas dos dois jogadores novos, dá o tamanho de cada uma, e `.tolist()` a converte numa lista.

Das 248 folhas da árvore sem limite, 233 têm um jogador só, e a previsão delas é o salário desse jogador. A árvore de três folhas prevê US$ 845.296 para o jogador com 130 rebatidas e US$ 402.776 para o com 60, a ordem que a figura das regiões sugere. A árvore sem limite prevê US$ 550.000 e US$ 700.000: mais para quem rebateu menos, porque cada uma das duas folhas tem um jogador só, e a previsão é o salário dele. Ela acerta o treino quase jogador a jogador, e a pergunta passa a ser outra: quanto ela erra num jogador que não viu?

## Podando a Árvore

> **📌 Nota**
>
> Esta seção corresponde à seção 8.1.1 de James et al. (2023).

Uma árvore que prevê o salário de jogadores de beisebol e cresce até cada folha ficar com um jogador só, ou só com jogadores de salário igual, erra pouco ou nada no treino. Quanto ela erra num jogador que não viu? E uma árvore pequena demais, com duas ou três folhas, perde o quê? Entre as duas pontas há muitas árvores de tamanho intermediário, e é entre elas que se escolhe.

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### Uma árvore grande

Os dados são os jogadores de `Hitters` com salário registrado, e a resposta `y` é o logaritmo do salário. A árvore pode perguntar sobre todas as colunas, não só `anos` e `rebatidas`, e três delas são de texto:

In [ ]:
hitters = pd.read_csv("dados/Hitters.csv").dropna()
print("jogadores:", len(hitters))
print(hitters[["liga", "divisao", "liga_seguinte"]].nunique())

> **🔧 Função**
>
> **`df[colunas].nunique()`** — o número de valores distintos de cada coluna da lista.

São 263 jogadores, e cada coluna de texto tem duas categorias. Cada uma vira uma coluna indicadora de 0 e 1, e os jogadores são divididos em duas metades, uma para treinar e outra para medir o erro de teste:

In [ ]:
y = np.log(hitters["salario"])
X = pd.get_dummies(hitters.drop(columns="salario"),
                   drop_first=True, dtype=float)
for coluna in X.columns:
    if coluna not in hitters.columns:
        print("indicadora:", coluna)
treino_X, teste_X, treino_y, teste_y = train_test_split(
    X, y, train_size=132, random_state=12)
print("colunas:", X.shape[1])
print("treino:", len(treino_X), "  teste:", len(teste_X))

> **🔧 Função**
>
> **`pd.get_dummies(df, drop_first, dtype)`** — troca cada coluna de texto por colunas indicadoras, uma por categoria, com 1 na linha que tem aquela categoria e 0 nas outras. `drop_first=True` descarta a primeira categoria de cada coluna, em ordem alfabética, que fica implícita nos zeros das outras; `dtype=float` grava 0,0 e 1,0 em vez de `False` e `True`.
>
> **`df.drop(columns=coluna)`** — a tabela sem a coluna dada. **`X.shape`** — o par (linhas, colunas); `X.shape[1]` é o número de colunas.
>
> **`train_test_split(X, y, train_size, random_state)`** — sorteia quais linhas vão para o treino e quais para o teste; `train_size=132` põe 132 linhas no treino e o resto no teste, e `random_state` fixa o sorteio.

As categorias que ficaram implícitas são as primeiras em ordem alfabética: `liga_Nacional` vale 0 para a liga Americana, `divisao_Oeste` vale 0 para a divisão Leste.

Sem nenhum limite, a árvore cresce até não ter mais o que dividir:

In [ ]:
grande = DecisionTreeRegressor(random_state=12).fit(treino_X, treino_y)
folha_grande = grande.apply(treino_X)
por_folha = pd.Series(folha_grande).value_counts()
distintos = treino_y.groupby(folha_grande).nunique()
print("folhas:", grande.get_n_leaves())
print("profundidade:", grande.get_depth())
print("folhas com 1 jogador:",
      numero_br(100 * (por_folha == 1).mean(), 1) + "%")
print("folha com salários diferentes:", bool((distintos > 1).any()))
mse_treino = mean_squared_error(treino_y, grande.predict(treino_X))
mse_teste = mean_squared_error(teste_y, grande.predict(teste_X))
mse_nulo = mean_squared_error(
    teste_y, np.full(len(teste_y), treino_y.mean()))
print("MSE de treino:", numero_br(mse_treino, 4))
print("MSE de teste:", numero_br(mse_teste, 4))
print("teste, média do treino:", numero_br(mse_nulo, 4))
print("teste < metade do da média:", bool(mse_teste < mse_nulo / 2))

> **🔧 Função**
>
> **`s.groupby(rotulos)`** — separa a série pelos `rotulos`, um por linha; aqui, a folha de cada jogador. Seguido de `.nunique()`, dá o número de valores distintos em cada grupo.
>
> **`mean_squared_error(y, previsto)`** — o erro quadrático médio: a média de $(y_i - \hat y_i)^2$.
>
> **`np.full(n, valor)`** — um array de `n` posições, todas com `valor`; aqui, a média do treino repetida para cada jogador do teste.

A árvore tem mais de uma centena de folhas para 132 jogadores de treino, e mais de 90% das folhas guardam um jogador só. O MSE de treino é 0,0000: cada jogador de treino recebe a média da sua folha, e nenhuma folha junta jogadores de salários diferentes. Numa folha de um jogador, a média é o próprio salário dele; nas outras, todos os jogadores ganham o mesmo, e a árvore parou de dividir porque não havia mais erro a reduzir. No teste, o MSE deixa de ser zero. Prever para todo jogador do teste a média do treino, sem árvore nenhuma, dá 0,8393, e a árvore grande fica abaixo da metade disso. Ela ajuda, mas o erro de treino zero não diz quanto: a árvore decorou os 132 jogadores.

Uma árvore menor teria menos variância, ao custo de algum viés. Um jeito de obtê-la seria parar de crescer quando nenhum corte reduzisse o RSS além de um limiar. Esse critério é míope: um corte que quase não reduz o RSS pode abrir caminho para um corte muito bom logo abaixo dele, e a árvore que parou antes nunca chega a vê-lo. A alternativa é crescer a árvore grande primeiro e depois cortar os galhos que não compensam, olhando a árvore inteira. É a **poda**.

### Poda por custo-complexidade

Podar uma árvore é escolher uma **subárvore**: juntar os descendentes de um nó interno numa folha só, a média de todos os jogadores que passavam por ele. A árvore grande, $T_0$, tem subárvores demais para estimar o erro de teste de cada uma. A poda precisa reduzir a lista a uma sequência curta de candidatas, e a saída lembra o lasso da seção 11.4: somar ao erro de treino uma penalidade pelo tamanho. No lasso, a penalidade era $\lambda\sum_j|\beta_j|$; na árvore, ela conta folhas, como o $d$ dos critérios da seção 11.2 contava preditores.

> **🔷 Conceito**
>
> Na **poda por custo-complexidade** (*cost complexity pruning*), para cada valor de um parâmetro de ajuste $\alpha \ge 0$, a subárvore $T \subset T_0$ escolhida é a que minimiza
>
> $$
> \begin{aligned}
> &\sum_{m=1}^{|T|} \sum_{i:\, x_i \in R_m} \left(y_i - \hat y_{R_m}\right)^2\\
> &\quad + \alpha |T|,
> \end{aligned}
> \tag{8.4}
> $$
>
> em que $|T|$ é o número de folhas de $T$, $R_m$ é o retângulo da $m$-ésima folha e $\hat y_{R_m}$ é a média da resposta dos jogadores de treino em $R_m$. A primeira parcela é o RSS de treino da subárvore.

Com $\alpha = 0$, (8.4) é só o RSS de treino, e a subárvore que o minimiza é a própria $T_0$. Cada folha a mais custa $\alpha$, e uma folha só vale a pena se reduzir o RSS em mais do que isso. Com $\alpha$ grande o bastante, sobra a raiz sozinha, que prevê a média de todos. Entre os dois extremos, quando $\alpha$ cresce, os galhos caem, e cai primeiro o que menos reduz o RSS por folha: por isso o método também se chama **poda do elo mais fraco** (*weakest link pruning*). As subárvores saem aninhadas, cada uma dentro da anterior, e a sequência inteira sai de uma conta só.

No `scikit-learn`, a conta é o `cost_complexity_pruning_path`, que devolve os valores de `ccp_alpha` em que um galho cai e o MSE de treino da subárvore que sobra em cada um. A escala é outra: o `scikit-learn` usa o MSE de treino, o RSS dividido pelo número $n$ de jogadores de treino, e minimiza $\text{MSE} + \texttt{ccp\_alpha}\,|T|$. Multiplicando tudo por $n$, fica (8.4) com $\alpha = n \times \texttt{ccp\_alpha}$.

In [ ]:
caminho = grande.cost_complexity_pruning_path(treino_X, treino_y)
alfas, mses = caminho.ccp_alphas, caminho.impurities
print("valores no caminho:", len(alfas))
print("raiz sozinha a partir de", numero_br(alfas[-1], 4))
print("valores repetidos:", bool((np.diff(alfas) < 1e-12).any()))

> **🔧 Função**
>
> **`arvore.cost_complexity_pruning_path(X, y)`** — a sequência de podas da árvore ajustada. `.ccp_alphas` são os valores de `ccp_alpha` em que um galho cai, em ordem crescente, começando em zero; `.impurities`, o MSE de treino da subárvore que sobra em cada um.
>
> **`np.diff(x)`** — a diferença entre cada elemento e o anterior; aqui, abaixo de $10^{-12}$ marca dois valores consecutivos iguais a menos do arredondamento.

Os `alfas` do caminho são os pontos de troca: exatamente num deles, a subárvore está passando de uma para a outra. Para ter uma subárvore de cada trecho, a grade toma um ponto entre cada par de valores consecutivos, a média geométrica dos dois, $\sqrt{a_k\, a_{k+1}}$, que fica no meio do intervalo em escala logarítmica. O primeiro ponto é o zero, e a grade para antes do último valor do caminho, a partir do qual sobra só a raiz.

A saída `valores repetidos: True`, do chunk do caminho, mostra que há valores consecutivos iguais a menos do arredondamento: aí o ponto cai em cima deles, e não dentro de um intervalo, e dois pontos da grade podem dar o mesmo tamanho. Cada ponto é uma árvore podada:

In [ ]:
grade = np.r_[0.0, np.sqrt(alfas[1:-1] * alfas[2:])]
tamanhos = pd.DataFrame({
    "ccp_alpha": grade,
    "folhas": [DecisionTreeRegressor(ccp_alpha=a, random_state=12)
               .fit(treino_X, treino_y).get_n_leaves() for a in grade],
})
print("pontos na grade:", len(grade))
print("tamanhos distintos:", tamanhos["folhas"].nunique())
print("em 0, a árvore inteira:",
      bool(tamanhos["folhas"].iloc[0] == grande.get_n_leaves()))
print("folhas nunca aumentam:",
      tamanhos["folhas"].is_monotonic_decreasing)
ausentes = sorted(set(range(2, 21)) - set(tamanhos["folhas"]))
print("tamanhos ausentes de 2 a 20:", ausentes)

> **🔧 Função**
>
> **`np.r_[a, b]`** — junta valores e arrays num array só. **`np.sqrt(x)`** — a raiz quadrada de cada elemento.
>
> **`DecisionTreeRegressor(ccp_alpha)`** — cresce a árvore e depois a poda com esse `ccp_alpha`, que vale $\alpha/n$ na escala de (8.4), com $n$ o número de linhas do ajuste.
>
> **`s.iloc[i]`** — o valor na posição `i` da série, contando do zero; `s.iloc[-1]` é o último.

Em `ccp_alpha` = 0 fica a árvore inteira, e ao longo da grade as folhas nunca aumentam. A grade tem menos tamanhos distintos que pontos, e não dá uma subárvore de cada tamanho: entre 2 e 20 folhas, falta a de 9. O que acontece na passagem por 10 folhas mostra por quê:

In [ ]:
i = tamanhos.index[tamanhos["folhas"] == 8][0]
antes, depois = [DecisionTreeRegressor(ccp_alpha=a, random_state=12)
                 .fit(treino_X, treino_y) for a in grade[i - 1:i + 1]]
pares = pd.DataFrame({"antes": antes.apply(treino_X),
                      "depois": depois.apply(treino_X)})
juntou = pares.groupby("depois")["antes"].nunique()
print("folhas:", antes.get_n_leaves(), "→", depois.get_n_leaves())
print("folhas novas:", (juntou > 1).sum())
print("folhas antigas que ela junta:", juntou.max())

> **🔧 Função**
>
> **`df.index[mascara][0]`** — o rótulo da primeira linha em que a `mascara` vale `True`; aqui, o primeiro ponto da grade com 8 folhas.

De 10 folhas, a poda seguinte vai direto a 8. Caiu um galho só: uma única folha da subárvore de 8 reúne mais de uma folha da de 10, três delas. O nó que virou folha tinha três folhas abaixo dele, e trocá-lo por uma folha tira duas de uma vez (10 − 3 + 1 = 8). É esse o mecanismo da poda: um nó interno inteiro vira folha, com a média de todos os jogadores que passavam por ele.

In [ ]:
# Figura: O número de folhas da subárvore podada contra `ccp_alpha`, os dois eixos em escala logarítmica. Cada degrau é uma poda; à esquerda de 0,0001, a curva continua subindo até as folhas da árvore inteira, em `ccp_alpha` = 0.
x_degraus = np.r_[alfas[1:], 1.0]
y_degraus = np.r_[tamanhos["folhas"].to_numpy()[1:], 1, 1]
fig, ax = plt.subplots(figsize=(4.8, 3.4))
ax.step(x_degraus, y_degraus, where="post", color="C0", linewidth=2)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1e-4, 1)
ax.set_ylim(0.8, 150)
ax.set_yticks([1, 2, 5, 10, 20, 50, 100])
ax.set_yticklabels(["1", "2", "5", "10", "20", "50", "100"])
ax.set_xlabel("ccp_alpha")
ax.set_ylabel("folhas")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`s.to_numpy()`** — os valores da série como array do `numpy`, sem os rótulos. **`ax.step(x, y, where="post")`** — uma linha em degraus: cada `y[i]` vale de `x[i]` até `x[i + 1]`.
>
> **`ax.set_xscale("log")`**, **`ax.set_yscale("log")`** — eixos em escala logarítmica, em que cada multiplicação por 10 ocupa o mesmo espaço. **`ax.set_yticks`**, **`ax.set_yticklabels`** — as posições das marcas do eixo e o texto de cada uma.

Que grandeza é o `ccp_alpha` de um degrau? O penúltimo galho a cair, antes do que deixa a raiz sozinha, é o que leva de 3 para 2 folhas. No ponto de troca, as duas subárvores empatam em $\text{MSE} + \texttt{ccp\_alpha}\,|T|$, e o `ccp_alpha` daquele ponto é exatamente quanto o MSE de treino sobe com a folha perdida:

In [ ]:
print("folhas:", tamanhos["folhas"].iloc[-2], "→",
      tamanhos["folhas"].iloc[-1])
print("MSE:", numero_br(mses[-3], 4), "→", numero_br(mses[-2], 4))
print("ccp_alpha:", numero_br(alfas[-2], 4))
print("diferença igual:",
      bool(np.isclose(mses[-2] - mses[-3], alfas[-2])))
print("α = 132 × ccp_alpha =", numero_br(132 * alfas[-2]))

> **🔧 Função**
>
> **`np.isclose(a, b)`** — `True` se `a` e `b` são iguais a menos do arredondamento de ponto flutuante.

O MSE de treino sobe de 0,2258 para 0,2935, e a diferença é o `ccp_alpha` do degrau, 0,0677. Em RSS, com os 132 jogadores de treino, a folha perdida custa 8,94: é o $\alpha$ de (8.4) em que perder essa folha passa a compensar.

A comparação das folhas de cada jogador nas duas subárvores diz qual galho cai nesse degrau:

In [ ]:
tres, duas = [DecisionTreeRegressor(ccp_alpha=a, random_state=12)
              .fit(treino_X, treino_y) for a in grade[-2:]]
folha_duas = duas.apply(treino_X)
pares = pd.DataFrame({"tres": tres.apply(treino_X),
                      "duas": folha_duas})
nova = pares.groupby("duas")["tres"].nunique().idxmax()
n_nova = duas.tree_.n_node_samples[nova]
no = np.flatnonzero((tres.tree_.feature >= 0)
                    & (tres.tree_.n_node_samples == n_nova))[0]
print("folhas:", tres.get_n_leaves(), "→", duas.get_n_leaves())
print("nó que virou folha:")
print(" ", X.columns[tres.tree_.feature[no]], "≤",
      numero_br(tres.tree_.threshold[no], 1))
print("jogadores nele:", n_nova)
print("média da folha nova:",
      numero_br(treino_y[folha_duas == nova].mean()))

> **🔧 Função**
>
> **`s.idxmax()`** — o rótulo do maior valor da série; aqui, a folha da subárvore de 2 que junta mais folhas da de 3. **`arvore.tree_.n_node_samples`** — para cada nó, o número de jogadores de treino que passam por ele. **`np.flatnonzero(mascara)`** — as posições em que a `mascara` vale `True`; aqui, a do nó interno da subárvore de 3 com o mesmo número de jogadores da folha nova, que só o nó juntado tem.

O nó que virou folha é o que pergunta se `impulsionadas_carreira` ≤ 70,5, com 54 jogadores de treino. As duas folhas abaixo dele viram uma só, que prevê para os 54 a média deles, 5,11. A figura desenha as duas subárvores com a função `em_portugues` da seção 12.1, com duas mudanças para as caixas caberem na largura da figura: os nomes de coluna terminados em `_carreira` quebram em duas linhas, e, nas folhas, que não têm pergunta (`≤`), o valor da média desce para a linha de baixo.

In [ ]:
def em_portugues(anotacoes):
    """Traduz e ajusta as caixas que o plot_tree desenha."""
    for anotacao in anotacoes:
        texto = (anotacao.get_text()
                 .replace("samples", "n").replace("value", "média")
                 .replace("<=", "≤")
                 .replace("True", "sim").replace("False", "não")
                 .replace("_carreira", "_\ncarreira"))
        if "≤" not in texto:
            texto = texto.replace("média = ", "média =\n")
        texto = re.sub(r"(\d)\.(\d)", r"\1,\2", texto)
        anotacao.set_text(texto)
        caixa = anotacao.get_bbox_patch()
        if caixa is not None:
            caixa.set(facecolor="white", edgecolor="0.45")

In [ ]:
# Figura: Uma poda do elo mais fraco, de 3 para 2 folhas. Em cima, a caixa laranja é o nó que pergunta se impulsionadas_carreira ≤ 70,5; embaixo, é a folha que ficou no lugar dele, com os mesmos 54 jogadores e a média 5,11 deles. O resto da árvore não muda.
fig, eixos = plt.subplots(2, 1, figsize=(4.8, 7.6), height_ratios=[3, 2])
for ax, arvore, destaque, titulo in [(eixos[0], tres, no, "antes: 3 folhas"),
                                    (eixos[1], duas, nova, "depois: 2 folhas")]:
    anotacoes = plot_tree(arvore, feature_names=list(X.columns),
                          impurity=False, rounded=True, precision=2,
                          fontsize=10, ax=ax)
    em_portugues(anotacoes)
    caixas = [a for a in anotacoes if a.get_bbox_patch() is not None]
    caixas[destaque].get_bbox_patch().set(edgecolor="C1", linewidth=2.5)
    ax.set_title(titulo)
plt.show()

> **🔧 Função**
>
> **`plt.subplots(2, 1, figsize, height_ratios)`** — dois painéis empilhados; `height_ratios=[3, 2]` dá ao de cima, com três níveis de caixas, mais altura que ao de baixo, com dois. **`a.get_bbox_patch().set(edgecolor, linewidth)`** — muda a cor e a espessura da borda da caixa; só as caixas dos nós têm borda, e na ordem dos números dos nós, por isso a lista `caixas` deixa de fora os rótulos "sim" e "não". **`ax.set_title(texto)`** — o título do painel.

### Escolher α por validação cruzada

A sequência de subárvores está pronta; falta escolher uma, e o erro de treino não serve, porque ele sempre prefere $T_0$. O procedimento completo tem quatro passos:

1. crescer uma árvore grande com os dados de treino;
2. podá-la por custo-complexidade, obtendo a melhor subárvore para cada $\alpha$;
3. escolher $\alpha$ por validação cruzada em $K$ grupos: para cada grupo, repetir os passos 1 e 2 com os outros $K - 1$, medir o erro de cada $\alpha$ no grupo deixado de fora, e tirar a média dos $K$ grupos; fica o $\alpha$ de menor erro médio;
4. devolver a subárvore do passo 2 correspondente a esse $\alpha$.

O `GridSearchCV`, que a seção 11.5 usou para escolher o $\lambda$ da ridge e do lasso, faz o passo 3 com a grade de `ccp_alpha`: em cada grupo, cada ajuste com um `ccp_alpha` cresce a árvore nos outros grupos e a poda, que são os passos 1 e 2 refeitos. Com 132 jogadores de treino, 6 grupos dão 22 jogadores cada um. A grade vai em ordem decrescente de `ccp_alpha`, da árvore menor para a maior: quando dois valores empatam no erro, o `GridSearchCV` fica com o primeiro da grade, e o empate se resolve a favor da árvore menor.

Antes da saída: a curva do erro de treino só desce quando a árvore ganha folhas. A de validação cruzada também?

In [ ]:
busca = GridSearchCV(
    DecisionTreeRegressor(random_state=12),
    {"ccp_alpha": grade[::-1]},
    cv=KFold(6, shuffle=True, random_state=12),
    scoring="neg_mean_squared_error",
)
busca.fit(treino_X, treino_y)
res = busca.cv_results_
erros = -np.array([res[f"split{k}_test_score"] for k in range(6)])
curvas = pd.DataFrame({
    "ccp_alpha": grade[::-1],
    "folhas": tamanhos["folhas"].to_numpy()[::-1],
    "cv": erros.mean(axis=0),
    "desvio": erros.std(axis=0, ddof=1) / np.sqrt(6),
})
curvas = curvas.drop_duplicates("folhas").set_index("folhas")
escolhida = busca.best_estimator_
k_min = curvas["cv"].idxmin()
teto = curvas.loc[k_min, "cv"] + curvas.loc[k_min, "desvio"]
print("folhas:", escolhida.get_n_leaves())
print("ccp_alpha:", numero_br(busca.best_params_["ccp_alpha"], 4))
print("α = 132 × ccp_alpha =",
      numero_br(132 * busca.best_params_["ccp_alpha"]))
print("CV:", numero_br(curvas.loc[k_min, "cv"], 4))
print("desvio:", numero_br(curvas.loc[k_min, "desvio"], 4))
segunda = curvas["cv"].nsmallest(2)
print("segunda:", segunda.index[1], "folhas,",
      numero_br(segunda.iloc[1], 4))
print("teto:", numero_br(teto, 4))
print("regra do um desvio:",
      curvas.index[curvas["cv"] <= teto].min(), "folhas")

> **🔧 Função**
>
> **`KFold(k, shuffle, random_state)`** — divide as linhas em `k` grupos; `shuffle=True` sorteia quem vai para cada grupo, e `random_state` fixa o sorteio.
>
> **`GridSearchCV(estimador, grade, cv, scoring)`** — o erro de validação cruzada do `estimador` em cada valor da `grade`, um dicionário `{"parametro": valores}`, com os grupos de `cv`. `scoring="neg_mean_squared_error"` mede o MSE com o sinal trocado, porque a busca maximiza a pontuação.
>
> **`busca.cv_results_`** — as pontuações da busca; em `f"split{k}_test_score"`, a do grupo `k`, uma por valor da grade. **`busca.best_params_`** — o valor da grade de menor erro. **`busca.best_estimator_`** — o modelo reajustado com todas as linhas de treino e esse valor.
>
> **`np.array(lista)`** — monta um array com a lista; aqui, uma linha por grupo e uma coluna por valor da grade. **`erros.mean(axis=0)`**, **`erros.std(axis=0, ddof=1)`** — a média e o desvio padrão de cada coluna, isto é, de cada valor da grade sobre os 6 grupos; `axis=0` percorre as linhas, e `ddof=1` põe $n - 1$ no denominador.
>
> **`df.drop_duplicates("folhas")`** — fica com a primeira linha de cada número de folhas. **`df.set_index("folhas")`** — usa a coluna como rótulo das linhas. **`df.loc[rotulo, coluna]`** — o valor na linha de rótulo `rotulo` e na coluna dada. **`s.nsmallest(2)`** — os dois menores valores da série.

A validação cruzada escolhe a subárvore de 4 folhas, com erro de 0,2694, em `ccp_alpha` = 0,0343, isto é, $\alpha$ = 4,53 na escala de (8.4). A segunda melhor tem 7 folhas e erro de 0,2841. O desvio, $s/\sqrt{6}$ com $s$ o desvio padrão dos erros dos 6 grupos, é o da regra do um desvio da seção 11.2, e a regra também fica com 4 folhas: o teto é 0,3193, e nenhuma subárvore menor fica abaixo dele.

Dentro de cada grupo, a árvore é outra, crescida com 110 jogadores (132 − 22). Como o `ccp_alpha` é $\alpha/n$ com o $n$ de cada ajuste, ali o mesmo `ccp_alpha` corresponde a $\alpha = 110 \times$ `ccp_alpha` em (8.4), e pode deixar a árvore com um número de folhas diferente. As folhas que a figura abaixo põe no eixo horizontal são as da subárvore com aquele `ccp_alpha` nos 132 jogadores de treino. A mesma figura traz o erro de treino e o de teste de cada subárvore. A curva de teste está ali só para julgar a escolha: nenhum ponto dela entrou na decisão, que a validação cruzada tomou com os 132 jogadores de treino.

In [ ]:
for k, linha in curvas.iterrows():
    podada = DecisionTreeRegressor(ccp_alpha=linha["ccp_alpha"],
                                   random_state=12).fit(treino_X, treino_y)
    curvas.loc[k, "treino"] = mean_squared_error(
        treino_y, podada.predict(treino_X))
    curvas.loc[k, "teste"] = mean_squared_error(
        teste_y, podada.predict(teste_X))
mse_escolhida = mean_squared_error(teste_y, escolhida.predict(teste_X))
print("MSE de teste da escolhida:", numero_br(mse_escolhida, 4))
print("abaixo do da grande:", bool(mse_escolhida < mse_teste))
print("menor teste em:", curvas["teste"].idxmin(), "folhas")
de_2_a_20 = curvas.loc[curvas.index <= 20].sort_index()
print("treino só desce:",
      de_2_a_20["treino"].is_monotonic_decreasing)
print("teste acima da CV:",
      (de_2_a_20["teste"] > de_2_a_20["cv"]).sum(), "de", len(de_2_a_20))

> **🔧 Função**
>
> **`df.iterrows()`** — percorre a tabela uma linha por vez, devolvendo o rótulo (aqui, o número de folhas) e a linha. **`df.loc[k, "treino"] = valor`** — grava o valor na linha `k` da coluna, criando a coluna se ela não existe. **`df.loc[mascara]`** — só as linhas em que a `mascara` vale `True`. **`df.sort_index()`** — as linhas em ordem crescente de rótulo.

In [ ]:
# Figura: O MSE de treino (cinza), de validação cruzada em 6 grupos (azul, com a faixa de um desvio para cima e para baixo) e de teste (laranja) das subárvores podadas, contra o número de folhas, de 2 a 20. O x marca o menor erro de validação cruzada.
ate_20 = curvas.loc[curvas.index <= 20].sort_index()
fig, ax = plt.subplots(figsize=(4.8, 3.6))
ax.plot(ate_20.index, ate_20["treino"], color="0.45", linewidth=2,
        label="treino")
ax.fill_between(ate_20.index, ate_20["cv"] - ate_20["desvio"],
                ate_20["cv"] + ate_20["desvio"], color="C0", alpha=0.3,
                linewidth=0)
ax.plot(ate_20.index, ate_20["cv"], color="C0", linewidth=2,
        label="validação cruzada")
ax.plot(ate_20.index, ate_20["teste"], color="C1", linewidth=2,
        label="teste")
ax.scatter(k_min, curvas.loc[k_min, "cv"], marker="x", s=80,
           color="C0", linewidths=2.5, zorder=3, label="_nolegend_")
ax.set_xlim(2, 20)
ax.set_ylim(0, 0.5)
ax.yaxis.set_major_formatter(lambda valor, _: f"{valor:.1f}".replace(".", ","))
ax.set_xlabel("folhas")
ax.set_ylabel("MSE")
ax.legend(loc="center right", bbox_to_anchor=(1, 0.36))
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.fill_between(x, abaixo, acima, color, alpha)`** — pinta a faixa entre as curvas `abaixo` e `acima`; `alpha` é a opacidade. **`ax.legend(loc, bbox_to_anchor)`** — a legenda, com o canto `loc` preso ao ponto `bbox_to_anchor`, em frações do painel. **`ax.yaxis.set_major_formatter(funcao)`** — escreve cada marca do eixo vertical com a função dada, que recebe o valor e a posição; aqui, com uma casa e vírgula decimal.

O erro de treino só desce. O de validação cruzada tem o mínimo em 4 folhas e fica acima dele com mais folhas: as folhas a mais passam a acompanhar o ruído dos jogadores de treino, e a árvore pequena demais, com 2 ou 3 folhas, junta jogadores de salários muito diferentes. O erro de teste da subárvore escolhida é 0,3630, abaixo do da árvore grande e bem abaixo do 0,8393 da média do treino, e o menor erro de teste entre as subárvores também está em 4 folhas. A curva de teste fica acima da de validação cruzada em 17 dos 18 tamanhos da figura, e o 0,2694 da escolhida é o menor de muitos erros estimados com os mesmos 132 jogadores, por isso tende a ser otimista.

### A árvore podada

A figura abaixo desenha a subárvore de 4 folhas escolhida, reajustada com os 132 jogadores de treino:

In [ ]:
# Figura: A subárvore de 4 folhas escolhida por validação cruzada. Cada caixa mostra a pergunta, o número de jogadores de treino (n) e a média do logaritmo do salário deles; o ramo "sim", à esquerda, é o da condição verdadeira.
fig, ax = plt.subplots(figsize=(4.8, 4.2))
caixas = plot_tree(escolhida, feature_names=list(X.columns),
                   impurity=False, rounded=True, precision=2,
                   fontsize=10, ax=ax)
em_portugues(caixas)
plt.show()

O chunk abaixo imprime o salário típico de cada folha, da esquerda para a direita, que é a ordem dos números dos nós: a exponencial da média do logaritmo, que é a média geométrica dos salários da folha, vezes 1.000, porque o salário está em milhares de dólares. Depois vêm a pergunta de cada nó interno e a correlação entre `anos` e `vezes_ao_bastao_carreira`:

In [ ]:
folha = escolhida.apply(treino_X)
media_log = treino_y.groupby(folha).mean()
for i, valor in enumerate(media_log, start=1):
    print(f"R{i}: {numero_br(valor)} → US$",
          numero_br(np.exp(valor) * 1000, 0))
no = escolhida.tree_
for f, corte in zip(no.feature, no.threshold):
    if f >= 0:
        print(X.columns[f], "≤", numero_br(corte, 1))
print("correlação anos × carreira:", numero_br(
    hitters["anos"].corr(hitters["vezes_ao_bastao_carreira"]), 3))

> **🔧 Função**
>
> **`s.corr(outra)`** — a correlação entre as duas séries.

A primeira pergunta é sobre as vezes ao bastão na carreira inteira, com corte em 1.452,0. É uma medida de tempo de liga: a correlação dela com `anos` é de 0,916, e ela faz aqui o papel que `anos` fez na árvore da seção 12.1. Entre os jogadores com menos tempo, a árvore pergunta pelas corridas impulsionadas na carreira, com corte em 70,5, e os dois grupos ficam com salário típico de US$ 103.316 e US$ 235.202. Entre os mais experientes, a pergunta é sobre as rebatidas da temporada anterior, com corte em 122,5, e os típicos são US$ 469.377 e US$ 925.119. A árvore grande tinha mais de uma centena de folhas; a podada tem 4, e erra menos no teste que ela.

Com outros 132 jogadores no treino, a validação cruzada escolheria o mesmo tamanho, e a primeira pergunta seria sobre a mesma coluna? A função abaixo repete a divisão, o crescimento, a poda e a escolha por validação cruzada, e o chunk a roda com cinco sementes no sorteio do treino, de 1 a 5:

In [ ]:
def poda_por_cv(semente):
    """Divide, cresce, poda e escolhe por validação cruzada."""
    tr_X, _, tr_y, _ = train_test_split(X, y, train_size=132,
                                        random_state=semente)
    arvore = DecisionTreeRegressor(random_state=12).fit(tr_X, tr_y)
    a = arvore.cost_complexity_pruning_path(tr_X, tr_y).ccp_alphas
    g = np.r_[0.0, np.sqrt(a[1:-1] * a[2:])]
    b = GridSearchCV(DecisionTreeRegressor(random_state=12),
                     {"ccp_alpha": g[::-1]},
                     cv=KFold(6, shuffle=True, random_state=12),
                     scoring="neg_mean_squared_error").fit(tr_X, tr_y)
    melhor = b.best_estimator_
    return melhor.get_n_leaves(), X.columns[melhor.tree_.feature[0]]

outras = pd.DataFrame([poda_por_cv(s) for s in range(1, 6)],
                      columns=["folhas", "raiz"], index=range(1, 6))
print(outras)
print("folhas de", outras["folhas"].min(), "a", outras["folhas"].max())
print("raiz em vezes ao bastão:",
      (outras["raiz"] == "vezes_ao_bastao_carreira").sum(), "de 5")

> **🔧 Função**
>
> **`pd.DataFrame(linhas, columns, index)`** — monta uma tabela a partir de uma lista de linhas, com os nomes de coluna e os rótulos dados; aqui, uma linha por semente.

Com as sementes de 1 a 5, a validação cruzada escolhe de 3 a 18 folhas. A coluna da raiz muda menos: é `vezes_ao_bastao_carreira` em 4 das 5 divisões. O tamanho da árvore podada depende muito de quais 132 jogadores caíram no treino, e a poda reduz a variância da árvore grande, mas não a elimina. Uma saída é não depender de uma árvore só: crescer muitas, cada uma com uma amostra diferente dos jogadores, e juntar as previsões delas, que é o assunto do capítulo 13.

## Árvores de Classificação

> **📌 Nota**
>
> Esta seção corresponde à seção 8.1.2 de James et al. (2023).

Um paciente chega com dor no peito, e um exame de angiografia vai dizer se ele tem doença cardíaca. Antes do exame, com a idade, o colesterol, a frequência cardíaca máxima e outras medidas simples, dá para prever o resultado com algumas perguntas de sim ou não? O que se quer prever é uma classe, "sim" ou "não": a folha não tem média a ajustar, e o RSS da árvore de regressão deixa de servir para escolher o corte.

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_val_predict)
from sklearn.tree import DecisionTreeClassifier, plot_tree

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### Os pacientes

O conjunto `Heart` tem 13 medidas de pacientes que chegaram a um hospital com dor no peito, e a resposta `doenca_cardiaca`, o resultado da angiografia. Algumas medidas são números, como `idade`, `colesterol` e `freq_cardiaca_max`; outras são códigos, como `vasos_coloridos`, o número de vasos principais do coração que aparecem coloridos num exame de fluoroscopia; e duas são de texto: `dor_no_peito`, o tipo da dor, e `talio`, o resultado de um exame de imagem do coração com tálio.

In [ ]:
coracao = pd.read_csv("dados/Heart.csv")
print("pacientes:", len(coracao))
vazios = coracao.isna().sum()
print(vazios[vazios > 0])
coracao = coracao.dropna()
print("depois do dropna:", len(coracao))
y = coracao["doenca_cardiaca"]
print(y.value_counts())
print("erro de prever sempre 'não':",
      numero_br((y != "não").mean(), 4))

Dos 303 pacientes, seis têm alguma medida vazia, quatro em `vasos_coloridos` e dois em `talio`, e ficam de fora: sobram 297. São 160 sem doença e 137 com. Um classificador que dissesse "não" para todo paciente, sem olhar medida nenhuma, erraria os 137 doentes, uma fração de 0,4613. É esse o número que uma árvore precisa bater.

As duas colunas de texto viram colunas indicadoras com **todas** as categorias, sem o `drop_first` da seção 12.2:

In [ ]:
X = pd.get_dummies(coracao.drop(columns="doenca_cardiaca"), dtype=float)
print("medidas:", coracao.shape[1] - 1)
print("colunas:", X.shape[1])
for coluna in X.columns:
    if coluna not in coracao.columns:
        print(" ", coluna)

Numa regressão linear, a categoria descartada fica implícita nos zeros das outras e evita uma coluna que é combinação das demais. Numa árvore, a coluna a mais não atrapalha, e cada indicadora é uma pergunta possível: "o tálio deu normal?" é uma pergunta, "a dor é assintomática?" é outra. Com a primeira categoria descartada, a árvore não poderia perguntar diretamente por ela. Cada corte numa indicadora deixa uma categoria sozinha de um lado e todas as outras do outro. Com as 3 categorias de `talio`, isso cobre todas as divisões possíveis; com as 4 de `dor_no_peito`, nenhum corte manda duas categorias para cada lado. As 13 medidas viram 18 colunas: as 11 numéricas, 4 de `dor_no_peito` e 3 de `talio`.

### Três medidas de impureza

Uma árvore de classificação cresce como a de regressão: divisão binária recursiva, um corte de cada vez, o melhor naquele passo. O que muda é o que a folha prevê e como se mede a qualidade de um corte. A folha prevê a **classe mais comum** entre os pacientes de treino que caem nela. E, além da classe, ela guarda as proporções: uma folha quase só de doentes e outra com pouco mais da metade de doentes preveem as duas "sim", mas não com a mesma segurança.

Seja $\hat p_{mk}$ a proporção de pacientes de treino da região $R_m$ que são da classe $k$, entre $K$ classes; aqui, $K = 2$. O candidato mais direto para medir um corte é a **taxa de erro de classificação** da região, a fração que não é da classe mais comum:

$$
E = 1 - \max_k \hat p_{mk}. \tag{8.5}
$$

Na raiz, com os 297 pacientes, $\hat p$ vale 160/297 para "não" e 137/297 para "sim", e $E$ é 0,4613, o erro de prever sempre "não". O erro é um critério ruim para crescer a árvore, e a subseção seguinte mostra por quê. Duas medidas o substituem. O **índice de Gini** é

$$
G = \sum_{k=1}^{K} \hat p_{mk}\,(1 - \hat p_{mk}), \tag{8.6}
$$

e a **entropia** é

$$
D = -\sum_{k=1}^{K} \hat p_{mk} \log \hat p_{mk}, \tag{8.7}
$$

com $\log$ o logaritmo natural e $0 \log 0 = 0$. Os três valem zero numa região **pura**, em que todos os pacientes são da mesma classe, e crescem à medida que as classes se misturam. Por isso se chamam medidas de **impureza**. O chunk abaixo calcula as três medidas na raiz do `Heart` e numa região meio a meio, que é a mistura máxima com duas classes:

In [ ]:
p = y.value_counts(normalize=True).to_numpy()
print("p̂:", numero_br(p[0], 4), "e", numero_br(p[1], 4))
print("E:", numero_br(1 - p.max(), 4))
print("G:", numero_br((p * (1 - p)).sum(), 4))
print("D:", numero_br(-(p * np.log(p)).sum(), 4))
meio = np.array([0.5, 0.5])
print("G meio a meio:", numero_br((meio * (1 - meio)).sum(), 4))
print("D meio a meio:", numero_br(-(meio * np.log(meio)).sum(), 4))

> **🔧 Função**
>
> **`s.value_counts(normalize=True)`** — a fração de cada valor da série, em vez da contagem.

Na raiz, $G$ = 0,4970 e $D$ = 0,6901 ficam logo abaixo dos valores de uma região meio a meio, 0,5000 e 0,6931: com 160 pacientes de um lado e 137 do outro, a raiz é quase tão misturada quanto pode ser.

Com duas classes, $\hat p$ de uma determina o da outra, e as três medidas são funções de um número só. A figura abaixo as desenha com $\hat p$ indo de 0 a 1:

In [ ]:
# Figura: As três medidas de impureza de uma região com duas classes, contra a proporção $\hat p$ de uma delas: erro de classificação (cinza), índice de Gini (azul) e entropia (verde). As três valem zero numa região pura e têm o máximo em $\hat p = 0{,}5$.
p_grade = np.linspace(0.001, 0.999, 500)
duas = np.column_stack([p_grade, 1 - p_grade])
medidas = {
    "erro": 1 - duas.max(axis=1),
    "Gini": (duas * (1 - duas)).sum(axis=1),
    "entropia": -(duas * np.log(duas)).sum(axis=1),
}
cores = {"erro": "0.45", "Gini": "C0", "entropia": "C2"}
fig, ax = plt.subplots(figsize=(4.8, 3.4))
for nome, valores in medidas.items():
    ax.plot(p_grade, valores, color=cores[nome], linewidth=2, label=nome)
ax.set_xlim(0, 1)
ax.set_ylim(0, 0.75)
virgula = lambda valor, _: f"{valor:.1f}".replace(".", ",")
ax.xaxis.set_major_formatter(virgula)
ax.yaxis.set_major_formatter(virgula)
ax.set_xlabel("$\\hat p$")
ax.set_ylabel("impureza")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()
print("máximo da entropia:", numero_br(medidas["entropia"].max(), 4))
for nome in ["Gini", "entropia"]:
    print(nome, "acima do erro na grade:",
          bool((medidas[nome] > medidas["erro"]).all()))
print("0,5 na grade:", bool(np.isin(0.5, p_grade)))
print("em p̂ = 0,5, Gini e erro:", numero_br(2 * 0.5 * 0.5, 1),
      "e", numero_br(1 - 0.5, 1))

> **🔧 Função**
>
> **`np.linspace(inicio, fim, n)`** — `n` números igualmente espaçados de `inicio` a `fim`. Aqui a grade termina a um milésimo das pontas, porque $\log 0$ não existe. **`np.column_stack([a, b])`** — junta os arrays como colunas de uma matriz. **`duas.max(axis=1)`**, **`.sum(axis=1)`** — o maior valor e a soma de cada linha. **`np.isin(valor, array)`** — se o valor aparece no array.

O erro é feito de dois segmentos de reta, e o Gini e a entropia são curvas arredondadas, que ficam acima do erro em toda a grade. A grade não passa por 0,5, e ali o Gini encontra o erro: os dois valem 0,5. Nas pontas, as três medidas valem zero. A entropia tem outra escala, com o máximo em 0,6931, e uma forma parecida com a do Gini: as duas são arredondadas, sem trecho reto. Para escolher o corte, é essa diferença de forma que decide, mais que a altura das curvas.

### Um corte, a mesma classe

Um corte troca um nó, o **pai**, por dois **filhos**, e a impureza depois do corte é a média das impurezas dos filhos, ponderada pelo número de pacientes de cada um. O que acontece com essa média depende da forma da curva, em quatro passos:

1. O $\hat p$ do pai é a média ponderada dos $\hat p$ dos filhos: os doentes do pai são os doentes de um filho somados aos do outro.
2. No gráfico de uma impureza contra $\hat p$, cada filho é um ponto da curva. A impureza depois do corte fica no segmento que liga esses dois pontos, a **corda**, na altura do $\hat p$ do pai. A figura depois do exemplo abaixo desenha isso com os pacientes do `Heart`.
3. Se os dois filhos ficam do mesmo lado de 0,5, isto é, têm a mesma classe mais comum que o pai, os dois estão no mesmo trecho reto do erro, e a corda coincide com ele: o erro depois do corte é igual ao do pai, por mais que um dos filhos fique quase puro.
4. O Gini e a entropia não têm trecho reto: se os $\hat p$ dos filhos são diferentes, a corda passa abaixo da curva, e a impureza depois do corte é menor que a do pai. Com $\hat p$ iguais, os dois filhos caem no mesmo ponto, e o corte não muda nenhuma das três medidas.

Com duas classes e $\hat p$ a fração de doentes, as três medidas ficam $E = \min(\hat p, 1 - \hat p)$, $G = 2\hat p(1 - \hat p)$ e $D = -\hat p \log \hat p - (1 - \hat p)\log(1 - \hat p)$. Um exemplo com o `Heart`: os pacientes com o exame de tálio alterado e dor no peito assintomática, divididos pela `depressao_st`, a depressão do segmento ST do eletrocardiograma durante o esforço, em até 0,5 e acima de 0,5.

In [ ]:
def no(pacientes):
    """Tamanho, doentes, p̂ e as três impurezas de um nó."""
    n = len(pacientes)
    doentes = (pacientes["doenca_cardiaca"] == "sim").sum()
    p = doentes / n
    return {"n": n, "doentes": doentes, "p̂": p, "E": min(p, 1 - p),
            "G": 2 * p * (1 - p),
            "D": -(p * np.log(p) + (1 - p) * np.log(1 - p))}

grupo = coracao[(coracao["talio"] != "normal")
                & (coracao["dor_no_peito"] == "assintomática")]
baixa = grupo["depressao_st"] <= 0.5
corte = pd.DataFrame([no(grupo), no(grupo[baixa]), no(grupo[~baixa])],
                     index=["pai", "ST ≤ 0,5", "ST > 0,5"])
filhos = corte.iloc[1:]
for medida in ["p̂", "E", "G", "D"]:
    depois = np.average(filhos[medida], weights=filhos["n"])
    print(f"{medida}: pai {numero_br(corte.loc['pai', medida], 4)},",
          f"filhos {numero_br(depois, 4)}")
(corte.style.format("{:.0f}", subset=["n", "doentes"])
 .format(lambda v: numero_br(v, 3), subset=["p̂", "E", "G", "D"]))

> **🔧 Função**
>
> **`df[mascara]`** e **`df[~mascara]`** — as linhas em que a máscara vale `True` e as em que vale `False`; `~` troca um pelo outro. **`np.average(valores, weights)`** — a média ponderada dos `valores`, com os pesos dados; aqui, o número de pacientes de cada filho. **`df.style.format(formato, subset)`** — como na seção 12.2, escreve os números das colunas em `subset` com o formato dado.

O pai tem 79 doentes em 89 pacientes, e os filhos, 13 em 21 e 66 em 68. Os três ficam acima de 0,5 e preveem "sim". A média ponderada dos $\hat p$ dos filhos dá o $\hat p$ do pai, 0,8876, como o passo 1 diz. O erro depois do corte é igual ao do pai, 0,1124: os 10 sem doença do pai (89 − 79) continuam errados, 8 de um lado (21 − 13) e 2 do outro (68 − 66). O Gini cai de 0,1995 para 0,1549, e a entropia, de 0,3514 para 0,2582. A figura abaixo desenha as cordas do erro e do Gini com esses números:

In [ ]:
# Figura: O corte dos 89 pacientes com tálio alterado e dor assintomática pela depressão do ST, visto no erro de classificação (em cima) e no índice de Gini (embaixo), contra a fração $\hat p$ de doentes, de 0,5 a 1. Os pontos laranja são os dois filhos, ligados pela corda tracejada; o círculo é o pai, sobre a curva; o x é a impureza depois do corte, sobre a corda, no $\hat p$ do pai. No erro, o x cai dentro do círculo; no Gini, cai abaixo dele.
curva = {"E": lambda q: np.minimum(q, 1 - q),
         "G": lambda q: 2 * q * (1 - q)}
nome_curva = {"E": "erro", "G": "Gini"}
cor_curva = {"E": "0.45", "G": "C0"}
q = np.linspace(0.5, 1, 200)
p_pai = corte.loc["pai", "p̂"]
fig, eixos = plt.subplots(2, 1, figsize=(4.2, 5.2), sharex=True)
for ax, medida in zip(eixos, ["E", "G"]):
    ax.plot(q, curva[medida](q), color=cor_curva[medida], linewidth=2)
    ax.plot(filhos["p̂"], filhos[medida], color="C1", linestyle="--",
            linewidth=1.5)
    ax.scatter(filhos["p̂"], filhos[medida], color="C1", s=45, zorder=3)
    depois = np.average(filhos[medida], weights=filhos["n"])
    ax.scatter(p_pai, corte.loc["pai", medida], s=110, facecolor="none",
               edgecolor=plt.rcParams["text.color"], linewidth=1.5,
               zorder=4)
    ax.scatter(p_pai, depois, marker="x", s=60, color="C1", linewidth=2,
               zorder=5)
    rotulo_pai = "pai = depois" if medida == "E" else "pai"
    ax.annotate(rotulo_pai, (p_pai, corte.loc["pai", medida]),
                xytext=(8, 6), textcoords="offset points")
    for (x_f, y_f), deslocamento in zip(filhos[["p̂", medida]].to_numpy(),
                                        [(-12, -16), (-42, -4)]):
        ax.annotate("filho", (x_f, y_f), xytext=deslocamento,
                    textcoords="offset points")
    ax.set_ylim(0, 0.55)
    ax.set_ylabel(nome_curva[medida])
eixos[1].annotate("depois", (p_pai, depois), xytext=(-46, -14),
                  textcoords="offset points")
virgula = lambda valor, _: f"{valor:.1f}".replace(".", ",")
for ax in eixos:
    ax.xaxis.set_major_formatter(virgula)
    ax.yaxis.set_major_formatter(virgula)
eixos[0].set_xlim(0.5, 1)
eixos[1].set_xlabel("$\\hat p$ de doentes")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.minimum(a, b)`** — o menor dos dois, elemento a elemento; aqui, $\min(\hat p, 1 - \hat p)$ ao longo da grade. **`plt.subplots(2, 1, figsize, sharex=True)`** — dois painéis empilhados com o mesmo eixo horizontal. **`ax.annotate(texto, ponto, xytext, textcoords)`** — escreve `texto` perto de `ponto`; com `textcoords="offset points"`, `xytext` é o deslocamento em pontos tipográficos a partir dele. **`plt.rcParams["text.color"]`** — a cor do texto definida pelo estilo; aqui, para o círculo acompanhar o tema claro ou escuro.

No erro, a corda é o próprio segmento de reta, e o corte não muda nada. No Gini, a corda passa por baixo da curva, e a distância entre o círculo e o x, de 0,1995 a 0,1549, é o quanto o corte melhora. Um filho com 66 doentes em 68 é um grupo em que a previsão "sim" é quase certa, e o Gini registra isso; o erro só muda quando a classe mais comum de um dos filhos troca.

> **🔷 Conceito**
>
> Uma **árvore de classificação** divide o espaço dos preditores em regiões retangulares, como a de regressão, e prevê para toda observação de $R_m$ a classe mais comum entre as observações de treino de $R_m$. Para escolher os cortes, usa o índice de Gini (8.6) ou a entropia (8.7), que são mais sensíveis à pureza dos nós que o erro de classificação (8.5).

### Podar com validação cruzada

O `DecisionTreeClassifier` cresce a árvore com o índice de Gini por padrão. Sem nenhum limite, ela divide até cada folha ficar pura:

In [ ]:
grande = DecisionTreeClassifier(random_state=12).fit(X, y)
print("classes:", list(grande.classes_))
print("folhas:", grande.get_n_leaves())
print("profundidade:", grande.get_depth())
print("erro de treino:", numero_br(1 - grande.score(X, y), 4))

> **🔧 Função**
>
> **`DecisionTreeClassifier(criterion, random_state, ccp_alpha)`** — a árvore de classificação. `criterion` é a medida de impureza que escolhe os cortes: `"gini"`, o padrão, ou `"entropy"`, a entropia com logaritmo na base 2, o que só muda a escala. `random_state` e `ccp_alpha` fazem o mesmo que no `DecisionTreeRegressor`. O `predict` devolve a classe mais comum da folha.
>
> **`arvore.classes_`** — as classes que a árvore conhece, em ordem alfabética; é a ordem das contagens e das proporções que ela guarda. **`arvore.score(X, y)`** — a fração de acertos, a **acurácia**; o erro de classificação é 1 menos ela.

Com erro de treino zero, a árvore grande decorou os 297 pacientes, e o remédio é o da seção 12.2: o caminho de poda, uma grade com um ponto dentro de cada intervalo, e a validação cruzada para escolher o `ccp_alpha`. No classificador, o custo que o `ccp_alpha` equilibra com o número de folhas é a impureza das folhas pelo critério da árvore, ponderada pelo número de pacientes de cada uma; com o critério padrão, é o índice de Gini, e não o erro de classificação. O último valor do caminho, a raiz sozinha, mostra isso:

In [ ]:
caminho = grande.cost_complexity_pruning_path(X, y)
alfas = caminho.ccp_alphas
print("valores no caminho:", len(alfas))
print("impureza da raiz sozinha:", numero_br(caminho.impurities[-1], 4))
print("igual ao Gini da raiz:",
      bool(np.isclose(caminho.impurities[-1], (p * (1 - p)).sum())))
grade = np.r_[0.0, np.sqrt(alfas[1:-1] * alfas[2:])]
print("pontos na grade:", len(grade))

A impureza da raiz sozinha é o Gini da raiz, e não o 0,4613 do erro. Para escolher o tamanho da árvore, quando o que se quer é acertar a classe, a medida natural é o erro de classificação, e é ele que a validação cruzada mede: `scoring="accuracy"` pontua cada `ccp_alpha` pela acurácia nos pacientes deixados de fora, e o erro é 1 menos ela. Os grupos são **estratificados**: cada um tem aproximadamente a mesma proporção de doentes que os 297. Como na seção 12.2, a grade vai em ordem decrescente de `ccp_alpha`, para que um empate no erro se resolva a favor da árvore menor.

In [ ]:
grupos = StratifiedKFold(10, shuffle=True, random_state=12)
busca = GridSearchCV(DecisionTreeClassifier(random_state=12),
                     {"ccp_alpha": grade[::-1]}, cv=grupos,
                     scoring="accuracy")
busca.fit(X, y)
res = busca.cv_results_
erros = 1 - np.array([res[f"split{k}_test_score"] for k in range(10)])
curvas = pd.DataFrame({"ccp_alpha": grade[::-1]})
for i, a in enumerate(curvas["ccp_alpha"]):
    podada = DecisionTreeClassifier(ccp_alpha=a, random_state=12).fit(X, y)
    curvas.loc[i, "folhas"] = podada.get_n_leaves()
    curvas.loc[i, "treino"] = 1 - podada.score(X, y)
curvas["cv"] = erros.mean(axis=0)
curvas["desvio"] = erros.std(axis=0, ddof=1) / np.sqrt(10)
curvas["folhas"] = curvas["folhas"].astype(int)
curvas = curvas.drop_duplicates("folhas").set_index("folhas")
escolhida = busca.best_estimator_
k_min = escolhida.get_n_leaves()
print("escolhida:", k_min, "folhas")
print("CV da escolhida:", numero_br(curvas.loc[k_min, "cv"], 4))
print("menor CV da grade:", numero_br(curvas["cv"].min(), 4))
iguais = curvas.loc[[6, 7], ["treino", "cv"]]
print("6 e 7 folhas empatam:",
      bool((iguais.iloc[0] == iguais.iloc[1]).all()))
ate_12 = curvas.loc[curvas.index <= 12, ["treino", "cv", "desvio"]]
ate_12.sort_index().style.format(lambda v: numero_br(v, 4))

> **🔧 Função**
>
> **`StratifiedKFold(k, shuffle, random_state)`** — como o `KFold`, divide as linhas em `k` grupos, mas sorteia de modo que cada grupo tenha aproximadamente a mesma proporção de cada classe que o conjunto inteiro.
>
> **`scoring="accuracy"`** — no `GridSearchCV`, pontua cada valor da grade pela fração de acertos no grupo deixado de fora. **`s.astype(int)`** — converte os valores da série para inteiros.

O `desvio` é $s/\sqrt{10}$, com $s$ o desvio padrão dos erros dos 10 grupos, como na seção 12.2.

A validação cruzada escolhe a árvore de 6 folhas, com erro de 0,1951, bem abaixo do 0,4613 de prever sempre "não". O 0,1951 é o menor de muitos erros estimados com os mesmos 297 pacientes, e por isso tende a ser otimista. As árvores de 6 e de 7 folhas empatam no erro de treino e no de validação cruzada, e é a grade decrescente que desempata a favor da de 6. O empate tem explicação, que a subseção "O empate entre 6 e 7 folhas", no fim, mostra.

Com mais folhas, o erro de treino não sobe. O de validação cruzada também cai sempre?

In [ ]:
# Figura: O erro de classificação de treino (cinza) e de validação cruzada em 10 grupos (azul, com a faixa de um desvio para cima e para baixo) das subárvores podadas, contra o número de folhas, de 2 a 25. O x marca a árvore escolhida.
ate_25 = curvas.loc[curvas.index <= 25].sort_index()
fig, ax = plt.subplots(figsize=(4.8, 3.6))
ax.plot(ate_25.index, ate_25["treino"], color="0.45", linewidth=2,
        label="treino")
ax.fill_between(ate_25.index, ate_25["cv"] - ate_25["desvio"],
                ate_25["cv"] + ate_25["desvio"], color="C0", alpha=0.3,
                linewidth=0)
ax.plot(ate_25.index, ate_25["cv"], color="C0", linewidth=2,
        label="validação cruzada")
ax.scatter(k_min, curvas.loc[k_min, "cv"], marker="x", s=80, color="C0",
           linewidths=2.5, zorder=3, label="_nolegend_")
ax.set_xlim(2, 25)
ax.set_ylim(0, 0.35)
ax.set_yticks([0, 0.1, 0.2, 0.3])
ax.yaxis.set_major_formatter(lambda valor, _: f"{valor:.1f}".replace(".", ","))
ax.set_xlabel("folhas")
ax.set_ylabel("erro de classificação")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

In [ ]:
de_2_a_25 = curvas.loc[curvas.index <= 25].sort_index()
print("treino nunca sobe:",
      de_2_a_25["treino"].is_monotonic_decreasing)
print("menor CV em:", de_2_a_25["cv"].idxmin(), "folhas")
cauda = de_2_a_25.loc[de_2_a_25.index >= 18, "cv"]
print("CV de 18 a 25 folhas:", numero_br(cauda.iloc[0], 4))
print("igual em todas:", bool((cauda == cauda.iloc[0]).all()))
print("acima da de 2 folhas:",
      bool(cauda.iloc[0] > de_2_a_25.loc[2, "cv"]))

O de validação cruzada tem o mínimo em 6 folhas, empatado com 7, e volta a subir: de 18 a 25 folhas, fica em 0,2756, acima até do 0,2725 da árvore de 2 folhas. É o desenho da seção 12.2, com o erro de classificação no lugar do MSE.

O critério de impureza é mais uma escolha do modelo, e a validação cruzada pode fazê-la como faz a do `ccp_alpha`. O chunk abaixo repete o caminho, a grade e a busca com `criterion="entropy"`, nos mesmos grupos:

In [ ]:
for criterio in ["gini", "entropy"]:
    arvore = DecisionTreeClassifier(criterion=criterio, random_state=12)
    a = arvore.fit(X, y).cost_complexity_pruning_path(X, y).ccp_alphas
    g = np.r_[0.0, np.sqrt(a[1:-1] * a[2:])]
    b = GridSearchCV(arvore, {"ccp_alpha": g[::-1]}, cv=grupos,
                     scoring="accuracy").fit(X, y)
    print(f"{criterio}: CV {numero_br(1 - b.best_score_, 4)},",
          b.best_estimator_.get_n_leaves(), "folhas")

> **🔧 Função**
>
> **`busca.best_score_`** — a pontuação média de validação cruzada do valor escolhido; aqui, a acurácia, e 1 menos ela é o erro.

Com a entropia, a validação cruzada fica com 8 folhas e erro de 0,2153, acima do 0,1951 do Gini. Nestes pacientes e nestes grupos, o Gini ganha; em outro conjunto de dados, pode ser o contrário, e escolher entre os dois pela validação cruzada soma mais uma escolha às que tornam o mínimo otimista.

Uma taxa de erro junta dois erros de custo muito diferente num exame cardíaco: mandar para casa um doente e mandar para o cateterismo quem não tem nada. O `cross_val_predict` guarda, para cada paciente, a previsão da árvore ajustada sem o grupo dele, e a matriz de confusão da seção 9.5 separa os dois erros:

In [ ]:
previsto = cross_val_predict(
    DecisionTreeClassifier(ccp_alpha=busca.best_params_["ccp_alpha"],
                           random_state=12), X, y, cv=grupos)
print("erro somado:", numero_br((previsto != y).mean(), 4))
pd.DataFrame(confusion_matrix(y, previsto, labels=["não", "sim"]),
             index=pd.Index(["não", "sim"], name="verdadeiro"),
             columns=pd.Index(["não", "sim"], name="previsto"))

> **🔧 Função**
>
> **`cross_val_predict(modelo, X, y, cv)`** — ajusta o `modelo` uma vez para cada grupo de `cv`, sem as linhas do grupo, e devolve a previsão de cada linha feita pela árvore que não a viu. **`confusion_matrix(verdadeiro, previsto, labels)`** — conta, para cada par de classes, quantas linhas da classe da linha foram previstas como a classe da coluna. **`busca.best_params_`** — o valor da grade escolhido.

Dos 137 doentes, 39 recebem "não" e 98 recebem "sim"; dos 160 sem doença, 19 recebem "sim". Somados, os erros da matriz dão 0,1953, e não o 0,1951 da validação cruzada, que é a média dos erros dos 10 grupos: 297 pacientes não se dividem em 10 grupos do mesmo tamanho, e a média dos grupos não pesa cada paciente igual. O `ccp_alpha` foi escolhido com esses mesmos grupos, e a matriz carrega o mesmo otimismo. Mesmo assim, 39 dos 137 doentes, mais de um em cada quatro, sairiam sem diagnóstico. Mandar para casa um doente é o mais grave dos dois erros. Baixar o limiar da probabilidade, como na seção 9.5, troca parte desses erros por alarmes falsos.

### A árvore escolhida

A figura adiante desenha a árvore escolhida. Com seis folhas lado a lado, as caixas precisam ser estreitas, e a função `em_portugues` da seção 12.2 muda para caber: nos nós internos, fica só a pergunta e o número de pacientes, com o limiar na linha de baixo da pergunta e os nomes `dor_no_peito_…` e `vasos_coloridos` quebrados em duas linhas; nas folhas, ficam as contagens das duas classes, uma por linha, e a classe que a folha prevê.

In [ ]:
def em_portugues(anotacoes):
    """Traduz e ajusta as caixas que o plot_tree desenha."""
    for anotacao in anotacoes:
        texto = anotacao.get_text()
        contagem = re.search(r"value = \[(\d+), (\d+)\]", texto)
        if "<=" not in texto and contagem:  # folha
            nao, sim = contagem.groups()
            classe = "sim" if int(sim) > int(nao) else "não"
            texto = f"{nao} não\n{sim} sim\nprevê {classe}"
        else:
            texto = ("\n".join(texto.split("\n")[:2])
                     .replace("samples", "n").replace(" <= ", "\n≤ ")
                     .replace("True", "sim").replace("False", "não")
                     .replace("peito_", "peito_\n")
                     .replace("vasos_", "vasos_\n"))
            texto = re.sub(r"(\d)\.(\d)", r"\1,\2", texto)
        anotacao.set_text(texto)
        caixa = anotacao.get_bbox_patch()
        if caixa is not None:
            caixa.set(facecolor="white", edgecolor="0.45",
                      boxstyle="round,pad=0.2")
            if "prevê" in texto:
                cor = "C1" if texto.endswith("sim") else "C0"
                caixa.set(edgecolor=cor, linewidth=2)

> **🔧 Função**
>
> **`re.search(padrao, texto)`** — procura o `padrao` no texto e devolve o trecho encontrado, ou `None` se não acha; os parênteses do padrão marcam as partes a guardar, e **`.groups()`** as devolve, aqui as duas contagens. **`texto.split("\n")`** e **`"\n".join(linhas[:2])`** — separam o texto da caixa em linhas e juntam de volta só as duas primeiras, a pergunta e o `n`. **`boxstyle="round,pad=0.2"`** — cantos arredondados e uma margem menor entre o texto e a borda.

As folhas ganham borda azul quando preveem "não" e laranja quando preveem "sim", além do texto "prevê", para que a classe não dependa só da cor.

In [ ]:
# Figura: A árvore de 6 folhas escolhida por validação cruzada. Nos nós internos, a pergunta e o número de pacientes (n); nas folhas, quantos pacientes sem e com doença caem ali e a classe prevista, com a borda azul quando a folha prevê "não" e laranja quando prevê "sim". O ramo "sim", à esquerda, é o da condição verdadeira: numa indicadora, "≤ 0,5" quer dizer que o paciente não tem aquela categoria.
fig, ax = plt.subplots(figsize=(4.2, 4.6))
ax.set_position([0, 0, 1, 1])
caixas = plot_tree(escolhida, feature_names=list(X.columns),
                   class_names=list(escolhida.classes_), impurity=False,
                   rounded=True, precision=2, fontsize=9, ax=ax)
em_portugues(caixas)
plt.show()

> **🔧 Função**
>
> **`plot_tree(..., class_names)`** — num classificador, `class_names` dá o nome de cada classe, na ordem de `classes_`. **`ax.set_position([0, 0, 1, 1])`** — faz o painel ocupar a figura inteira, sem margens, para as caixas terem mais espaço.

O chunk abaixo conta os pacientes de cada folha, numeradas da esquerda para a direita, e lista as colunas que a árvore usa:

In [ ]:
tabela = pd.crosstab(escolhida.apply(X), y).reset_index(drop=True)
tabela.index += 1
tabela["prevê"] = tabela[["não", "sim"]].idxmax(axis=1)
fracao = tabela["sim"] / (tabela["não"] + tabela["sim"])
print("mais doentes: folha", tabela["sim"].idxmax())
print("maior fração de doentes: folha", fracao.idxmax())
print("colunas usadas:")
for coluna in X.columns[escolhida.feature_importances_ > 0]:
    print(" ", coluna)
tabela

> **🔧 Função**
>
> **`pd.crosstab(linhas, colunas)`** — conta quantas vezes cada par de valores aparece; aqui, a folha de cada paciente (`apply`, da seção 12.1) contra a classe dele. **`df.reset_index(drop=True)`** — troca os rótulos das linhas por 0, 1, 2…, descartando os antigos, que eram os números dos nós. **`df.idxmax(axis=1)`** — o nome da coluna com o maior valor em cada linha, aqui a classe mais comum da folha. **`arvore.feature_importances_`** — um número por coluna, positivo só nas que aparecem em algum corte da árvore.

A primeira pergunta é sobre o exame de tálio. Como `talio_normal` vale 1 para quem teve o exame normal, o ramo "sim" de `talio_normal ≤ 0,5`, à esquerda, é o dos pacientes com algum defeito no exame. Entre eles, os que têm dor no peito assintomática formam a folha 3, com 79 doentes em 89, a folha com mais doentes e com a maior fração deles. São os 89 pacientes da figura das cordas, na subseção "Um corte, a mesma classe". Do outro lado, os pacientes com tálio normal e nenhum vaso colorido no exame (`vasos_coloridos ≤ 0,5`) formam a folha com 102 sem doença em 115. As mesmas perguntas voltam em lugares diferentes da árvore: `vasos_coloridos` e `dor_no_peito_assintomática` aparecem nos dois lados, em ordens diferentes.

Três das 13 medidas bastaram, e nenhuma das outras dez aparece na árvore, nem `idade` nem `colesterol`. Isso quer dizer que a idade não tem relação com a doença cardíaca? Não necessariamente: em cada nó, a árvore fica com a pergunta que mais reduz a impureza naquele passo, e uma coluna que carrega parte da informação de outra já usada pode nunca ser a melhor.

### O empate entre 6 e 7 folhas

A árvore de 7 folhas tem uma folha a mais que a de 6, e as duas empataram no erro de treino e no de validação cruzada. O chunk abaixo segue os 89 pacientes do corte das cordas nas duas árvores e conta, entre eles, de que lado de 0,5 fica a depressão do ST em cada folha da árvore de 7:

In [ ]:
sete = DecisionTreeClassifier(ccp_alpha=curvas.loc[7, "ccp_alpha"],
                              random_state=12).fit(X, y)
no_grupo = X.loc[grupo.index]
print("folhas deles na de 6:", len(set(escolhida.apply(no_grupo))))
print("folhas deles na de 7:", len(set(sete.apply(no_grupo))))
print("mesmas previsões nos 297:",
      bool((sete.predict(X) == escolhida.predict(X)).all()))
st_baixa = grupo["depressao_st"] <= 0.5
pd.crosstab(sete.apply(no_grupo),
            st_baixa.map({True: "≤ 0,5", False: "> 0,5"}),
            rownames=["nó na de 7"], colnames=["ST"])

> **🔧 Função**
>
> **`pd.crosstab(..., rownames, colnames)`** — os nomes que aparecem nos cabeçalhos das linhas e das colunas. **`s.map(dicionario)`** — troca cada valor da série pelo que o dicionário associa a ele.

Na árvore de 6 folhas, os 89 pacientes com tálio alterado e dor assintomática formam uma folha só, a folha 3, que também tem 89. Na de 7, eles se espalham por duas folhas, e a tabela mostra que cada uma recebe só pacientes de um lado do corte: 21 com ST até 0,5 e 68 acima. É a folha a mais, e o corte é o da figura das cordas. Os dois filhos preveem "sim", como o pai, e as duas árvores dão a mesma classe para todos os 297 pacientes; por isso empatam no erro de treino.

A validação cruzada não usa essas duas árvores: em cada grupo, cresce e poda uma árvore com os outros nove grupos, uma para cada `ccp_alpha`. O empate ali precisa de outra conta, grupo a grupo:

In [ ]:
alfa_6, alfa_7 = curvas.loc[[6, 7], "ccp_alpha"]
iguais = 0
for treino, fora in grupos.split(X, y):
    X_t, y_t, X_f = X.iloc[treino], y.iloc[treino], X.iloc[fora]
    seis_g = DecisionTreeClassifier(ccp_alpha=alfa_6, random_state=12)
    sete_g = DecisionTreeClassifier(ccp_alpha=alfa_7, random_state=12)
    previsoes_6 = seis_g.fit(X_t, y_t).predict(X_f)
    previsoes_7 = sete_g.fit(X_t, y_t).predict(X_f)
    iguais += int((previsoes_6 == previsoes_7).all())
print("grupos com previsões iguais:", iguais, "de 10")

> **🔧 Função**
>
> **`grupos.split(X, y)`** — os pares (linhas de treino, linhas deixadas de fora) de cada grupo, na mesma ordem que o `GridSearchCV` usa. **`X.iloc[linhas]`** — as linhas nas posições dadas.

Nos 10 grupos, os dois valores de `ccp_alpha` dão as mesmas previsões para os pacientes deixados de fora, e o erro de cada grupo é o mesmo nos dois. Se o corte a mais não muda nenhuma classe, para que ele serve? Para prever a classe, para nada. Para dizer quanta certeza se tem, o `predict_proba` mostra a diferença, num paciente de cada lado do corte:

In [ ]:
dois = [st_baixa[st_baixa].index[0], st_baixa[~st_baixa].index[0]]
print("ST:", numero_br(coracao.loc[dois[0], "depressao_st"], 1),
      "e", numero_br(coracao.loc[dois[1], "depressao_st"], 1))
print("classes:", list(sete.classes_))
for nome, arvore in [("6 folhas", escolhida), ("7 folhas", sete)]:
    sim = arvore.predict_proba(X.loc[dois])[:, 1]
    print(f"{nome}: {numero_br(sim[0], 3)} e {numero_br(sim[1], 3)}")

> **🔧 Função**
>
> **`arvore.predict_proba(X)`** — para cada linha, a fração de cada classe na folha em que ela cai, uma coluna por classe, na ordem de `classes_`; `[:, 1]` fica com a coluna de "sim".

Na árvore de 6 folhas, os dois pacientes recebem 0,888 de probabilidade de doença, a fração de doentes da folha de 89. Na de 7, o paciente com ST até 0,5 recebe 0,619, e o outro, 0,971, as frações das duas folhas novas. A previsão é "sim" nas duas árvores, com segurança muito diferente, e é isso que o Gini e a entropia enxergam e o erro de classificação não. Essas frações vêm dos mesmos pacientes com que a árvore escolheu o corte, e por isso tendem a ser mais extremas do que seriam em pacientes novos.

## Árvores contra Modelos Lineares

> **📌 Nota**
>
> Esta seção corresponde à seção 8.1.3 de James et al. (2023).

Um jogador de beisebol com 6 anos de liga faz uma rebatida a mais na temporada. Quanto muda o salário que um modelo prevê para ele? A regressão linear dá sempre a mesma resposta, qualquer que seja o número de rebatidas que ele já tinha. A árvore de regressão responde "nada" quase sempre e "um salto" quando a rebatida a mais faz o jogador atravessar um corte. As duas preveem $Y$ a partir de $X_1, \dots, X_p$, com duas suposições opostas sobre a forma de $f$. Na regressão linear, cada preditor soma a $f$ um efeito que não depende dos outros:

$$
f(X) = \beta_0 + \sum_{j=1}^{p} X_j \beta_j.
\tag{8.8}
$$

Na árvore, $f$ é constante dentro de cada caixa:

$$
f(X) = \sum_{m=1}^{M} c_m\, 1(X \in R_m),
\tag{8.9}
$$

em que $R_1, \dots, R_M$ são as regiões retangulares das folhas, as mesmas da seção 12.1, lá contadas por $J$ e aqui por $M$: não se sobrepõem e cobrem o espaço dos preditores inteiro. O indicador $1(X \in R_m)$ vale 1 quando $X$ cai em $R_m$ e 0 quando não cai. Para cada $X$, só um dos indicadores vale 1, e $f(X)$ é o $c_m$ da região em que $X$ cai: na árvore de regressão, a média $\hat y_{R_m}$ da resposta dos dados de treino daquela folha. Qual das duas formas descreve melhor um conjunto de dados?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import (GridSearchCV, KFold, StratifiedKFold,
                                     cross_val_score)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

plt.style.use("estilo-figuras.mplstyle")

Os números saem com vírgula decimal e ponto de milhar:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### As duas formas nos mesmos jogadores

Os jogadores de `Hitters` com salário registrado, o logaritmo do salário como resposta e as duas colunas da seção 12.1, `anos` e `rebatidas`, bastam para ver as duas formas lado a lado. O chunk ajusta a regressão linear e a árvore de três folhas e confere que a soma (8.9), com as médias das folhas no lugar de $c_m$, reproduz as previsões da árvore:

In [ ]:
hitters = pd.read_csv("dados/Hitters.csv").dropna()
y = np.log(hitters["salario"])
X2 = hitters[["anos", "rebatidas"]]
linear = LinearRegression().fit(X2, y)
print("β0:", numero_br(linear.intercept_, 3))
for coluna, beta in zip(X2.columns, linear.coef_):
    print(f"β de {coluna}: {numero_br(beta, 4)}")
arvore = DecisionTreeRegressor(max_leaf_nodes=3, random_state=12)
folha = arvore.fit(X2, y).apply(X2)
c = y.groupby(folha).mean()
print("c_m:", ", ".join(numero_br(v, 3) for v in c))
soma = sum(c[m] * (folha == m) for m in c.index)
print("(8.9) reproduz a árvore:",
      bool(np.allclose(soma, arvore.predict(X2))))
col_rebatidas = list(X2.columns).index("rebatidas")
corte = arvore.tree_.threshold[arvore.tree_.feature == col_rebatidas]
print("corte em rebatidas:", numero_br(corte[0], 1))

> **🔧 Função**
>
> **`LinearRegression()`** — a regressão linear por mínimos quadrados. Depois do `fit`, **`linear.intercept_`** guarda $\hat\beta_0$, e **`linear.coef_`**, os $\hat\beta_j$, na ordem das colunas.
>
> **`folha == m`** — um vetor de `True` e `False`, com `True` nos jogadores da folha `m`: é o $1(X \in R_m)$ de (8.9), que na conta vale 1 e 0. **`np.allclose(a, b)`** — `True` se os dois arrays são iguais elemento a elemento, a menos do arredondamento.

Os três $c_m$ são as médias das folhas $R_1$, $R_2$ e $R_3$ da seção 12.1, e a árvore tem uma pergunta sobre rebatidas, com o corte impresso na última linha. As duas formas respondem de jeitos diferentes a uma mudança num preditor só. Para um jogador com 6 anos de liga, o que acontece com a previsão quando as rebatidas passam de 100 para 117, depois para 118 e depois para 140? Tente prever, para cada modelo, quais dos três passos mudam a previsão.

In [ ]:
novos = pd.DataFrame({"anos": 6, "rebatidas": [100, 117, 118, 140]})
previsoes = novos.assign(linear=linear.predict(novos),
                         árvore=arvore.predict(novos))
passo = previsoes[["linear", "árvore"]].diff().iloc[1:]
print("passos")
print("linear:", " / ".join(numero_br(v, 4) for v in passo["linear"]))
print("árvore:", " / ".join(numero_br(v, 4) for v in passo["árvore"]))
(previsoes.drop(columns="anos").style.hide(axis="index")
 .format(lambda v: numero_br(v, 4), subset=["linear", "árvore"]))

> **🔧 Função**
>
> No `pd.DataFrame`, um valor solto, como o 6, é repetido em todas as linhas. **`df.assign(nome=valores)`** — uma cópia da tabela com as colunas novas. **`df.diff()`** — a diferença entre cada linha e a anterior; a primeira fica vazia, e `.iloc[1:]` a descarta.

Na regressão linear, cada rebatida a mais soma o $\hat\beta$ de `rebatidas`, 0,0087, ao logaritmo do salário, seja qual for o número de rebatidas e o de anos. Os três passos da linear, 0,1473, 0,0087 e 0,1906, são proporcionais às 17, 1 e 22 rebatidas acrescentadas. Na árvore, as 17 rebatidas de 100 a 117 não mudam nada, e a 118.ª soma 0,7413 de uma vez, porque o jogador atravessa o corte de 117,5 e passa de $R_2$ para $R_3$. De 118 a 140, a previsão fica parada de novo. A figura mostra as duas previsões para todas as contagens de rebatidas, com os anos fixos em 6:

In [ ]:
# Figura: O logaritmo do salário previsto (salário em milhares de dólares) para um jogador com 6 anos de liga, conforme o número de rebatidas: a regressão linear é uma reta, e a árvore de três folhas, uma escada com um degrau só, no corte em rebatidas (linha pontilhada cinza). Os pontos marcam as quatro contagens da tabela: 100, 117, 118 e 140.
grade_reb = pd.DataFrame({"anos": 6, "rebatidas": np.arange(0, 241)})
fig, ax = plt.subplots(figsize=(4.2, 3.2))
ax.plot(grade_reb["rebatidas"], linear.predict(grade_reb), color="C0",
        label="linear")
ax.step(grade_reb["rebatidas"], arvore.predict(grade_reb), where="post",
        color="C1", label="árvore")
ax.scatter(previsoes["rebatidas"], previsoes["linear"], color="C0", s=20,
           zorder=3)
ax.scatter(previsoes["rebatidas"], previsoes["árvore"], color="C1", s=20,
           zorder=3)
ax.axvline(corte[0], color="0.45", linestyle=":", linewidth=1.2)
ax.set_xlim(0, 240)
ax.set_xlabel("rebatidas")
ax.set_ylabel("log do salário previsto")
ax.yaxis.set_major_formatter(lambda valor, _: f"{valor:.1f}".replace(".", ","))
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.arange(a, b)`** — os inteiros de `a` até `b - 1`; aqui, as 241 contagens de rebatidas de 0 a 240.

Se a relação entre os preditores e a resposta for próxima de uma soma de efeitos, como (8.8), a regressão linear aproveita essa estrutura, e a árvore, que só consegue imitá-la com muitos degraus, sai perdendo. Se a relação tiver limiares e interações, como (8.9), a árvore pode ganhar. Nenhuma das duas é melhor em geral, e quem decide, num conjunto de dados, é a estimativa do erro de teste.

### Duas fronteiras

Na classificação, o papel de (8.8) é da regressão logística: com duas classes, ela prevê a classe 1 quando a probabilidade estimada passa de 0,5, isto é, quando $\hat\beta_0 + \hat\beta_1 X_1 + \hat\beta_2 X_2 > 0$, e a fronteira entre as duas classes é a reta $\hat\beta_0 + \hat\beta_1 X_1 + \hat\beta_2 X_2 = 0$. A árvore de classificação divide o plano em caixas, e a fronteira dela é feita de segmentos paralelos aos eixos. Para ver o que cada uma faz com uma fronteira que não é a sua, o chunk abaixo sorteia 200 pontos de treino e 10.000 de teste no quadrado $[-2, 2] \times [-2, 2]$ e dá a eles duas classificações verdadeiras:

- **reta**: classe 1 quando $X_1 + X_2 > 0$, uma fronteira diagonal;
- **degraus**: classe 1 quando $X_1 < -0{,}5$ ou $X_2 > 0{,}5$, uma fronteira em L, com um segmento vertical e um horizontal.

Em cada classificação, cerca de 10% dos rótulos, sorteados, trocam de classe: é o ruído, que nenhum modelo prevê.

In [ ]:
rng = np.random.default_rng(12)
X = rng.uniform(-2, 2, size=(200, 2))
X_teste = rng.uniform(-2, 2, size=(10_000, 2))
troca = rng.random(200) < 0.1
troca_teste = rng.random(10_000) < 0.1

def reta(Z):
    return (Z[:, 0] + Z[:, 1] > 0).astype(int)

def degraus(Z):
    return ((Z[:, 0] < -0.5) | (Z[:, 1] > 0.5)).astype(int)

rotulos = {nome: (f(X) ^ troca, f(X_teste) ^ troca_teste)
           for nome, f in [("reta", reta), ("degraus", degraus)]}
print("trocados no treino:", troca.sum(), "de 200")
print("fração trocada no teste:", numero_br(troca_teste.mean(), 4))

> **🔧 Função**
>
> **`np.random.default_rng(semente)`** — um gerador de números aleatórios do `numpy`, com a semente dada. **`rng.uniform(a, b, size)`** — sorteios uniformes entre `a` e `b`, no formato `size`; aqui, 200 linhas de duas colunas. **`rng.random(n)`** — `n` sorteios uniformes entre 0 e 1; `< 0.1` marca, em média, um em cada dez.
>
> **`(condicao).astype(int)`** — converte `True` e `False` em 1 e 0. **`a ^ b`** — o "ou exclusivo": vale 1 quando só um dos dois vale 1. Com `troca`, inverte a classe exatamente nos pontos sorteados.

Um classificador que conhecesse a fronteira verdadeira erraria exatamente os pontos trocados: é o classificador de Bayes da seção 7.7, e a fração trocada no teste, 0,0991, é o erro dele nestes 10.000 pontos. Esse erro é a referência contra a qual os modelos se medem. As trocas são o erro irredutível desta classificação: nenhum modelo tem como prever o sorteio, e, a não ser por acaso num conjunto de teste finito, nenhum erra menos que o classificador de Bayes.

Os dois modelos são ajustados nos 200 pontos de treino, uma vez para cada classificação verdadeira. A logística vai sem penalização; a árvore é podada como na seção 12.3, com a grade de um ponto por intervalo do caminho, em ordem decrescente, e a validação cruzada estratificada em 10 grupos. O primeiro chunk ajusta e guarda os modelos e os erros de validação cruzada:

In [ ]:
grupos = StratifiedKFold(10, shuffle=True, random_state=12)

def arvore_podada(X, y):
    """Cresce, poda e escolhe o ccp_alpha por validação cruzada."""
    arvore = DecisionTreeClassifier(random_state=12)
    a = arvore.fit(X, y).cost_complexity_pruning_path(X, y).ccp_alphas
    grade = np.r_[0.0, np.sqrt(a[1:-1] * a[2:])]
    return GridSearchCV(arvore, {"ccp_alpha": grade[::-1]}, cv=grupos,
                        scoring="accuracy").fit(X, y)

modelos, buscas, erros_cv = {}, {}, {}
for nome, (y_s, y_teste) in rotulos.items():
    logistica = LogisticRegression(C=np.inf, solver="newton-cholesky",
                                   tol=1e-8)
    cv_log = cross_val_score(logistica, X, y_s, cv=grupos).mean()
    busca = buscas[nome] = arvore_podada(X, y_s)
    modelos[nome] = {"logística": logistica.fit(X, y_s),
                     "árvore": busca.best_estimator_}
    erros_cv[nome] = {"logística": 1 - cv_log,
                      "árvore": 1 - busca.best_score_}

> **🔧 Função**
>
> **`LogisticRegression(C, solver, tol)`** — a regressão logística. `C=np.inf` a ajusta sem penalização; `solver="newton-cholesky"` e `tol=1e-8` são o método de otimização e a tolerância para parar, que levam ao mesmo resultado em qualquer máquina.
>
> **`cross_val_score(estimador, X, y, cv)`** — ajusta e avalia o estimador em cada grupo de `cv` e devolve uma pontuação por grupo; num classificador, a pontuação padrão é a acurácia, e 1 menos a média dela é o erro de validação cruzada.

O segundo mede os dois modelos nos 10.000 pontos de teste e imprime os erros:

In [ ]:
erros = {}
for nome, (y_s, y_teste) in rotulos.items():
    erros[nome] = {m: 1 - modelo.score(X_teste, y_teste)
                   for m, modelo in modelos[nome].items()}
    print(f"{nome}:")
    for m in ["logística", "árvore"]:
        print(f"  {m}: CV {numero_br(erros_cv[nome][m], 4)},"
              f" teste {numero_br(erros[nome][m], 4)}")
    print("  folhas da árvore:", modelos[nome]["árvore"].get_n_leaves())
    print("  menor erro de teste:", min(erros[nome], key=erros[nome].get))

Antes da figura: a fronteira da árvore só tem segmentos horizontais e verticais. Como ela desenha a diagonal $X_1 + X_2 = 0$? E como a logística, que só desenha retas, desenha o L?

In [ ]:
# Figura: As regiões que cada modelo prevê: azul para a classe 0 e laranja para a classe 1, em tom mais fraco que o dos pontos, com os 200 pontos de treino na cor da classe deles, trocas incluídas. Na linha de cima, a fronteira verdadeira é a reta $X_1 + X_2 = 0$; na de baixo, o L de $X_1 = -0{,}5$ e $X_2 = 0{,}5$. A linha tracejada cinza é a fronteira verdadeira. À esquerda, a regressão logística; à direita, a árvore podada por validação cruzada. O número no título de cada painel é o erro de teste, nos 10.000 pontos de teste.
eixo = np.linspace(-2, 2, 200)
m1, m2 = np.meshgrid(eixo, eixo)
malha = np.column_stack([m1.ravel(), m2.ravel()])
verdadeira = {"reta": ([-2, 2], [2, -2]),
              "degraus": ([-0.5, -0.5, 2], [-2, 0.5, 0.5])}
cor_classe = np.array(["C0", "C1"])
fig, eixos = plt.subplots(2, 2, figsize=(3.6, 4.2), sharex=True,
                          sharey=True)
for linha, nome in zip(eixos, ["reta", "degraus"]):
    y_s = rotulos[nome][0]
    for ax, (tipo, modelo) in zip(linha, modelos[nome].items()):
        z = modelo.predict(malha).reshape(m1.shape)
        ax.contourf(m1, m2, z, levels=[-0.5, 0.5, 1.5],
                    colors=["C0", "C1"], alpha=0.3)
        ax.scatter(X[:, 0], X[:, 1], s=5, c=cor_classe[y_s], linewidths=0)
        ax.plot(*verdadeira[nome], color="0.45", linestyle="--",
                linewidth=1.5)
        ax.set_title(f"{tipo}\nteste {numero_br(erros[nome][tipo], 4)}",
                     fontsize=8)
        ax.set_aspect("equal")
        ax.set_xlim(-2, 2)
        ax.set_ylim(-2, 2)
        ax.set_xticks([-1, 0, 1])
        ax.set_yticks([-1, 0, 1])
    linha[0].set_ylabel(f"{nome}\n$X_2$")
for ax in eixos[1]:
    ax.set_xlabel("$X_1$")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.meshgrid(x, y)`** — as duas coordenadas de uma malha: cada ponto da grade 200 × 200 recebe o seu $X_1$ em `m1` e o seu $X_2$ em `m2`. **`m1.ravel()`** — a matriz achatada numa fila só, para o `np.column_stack` montar as duas colunas que os modelos esperam; **`.reshape(m1.shape)`** faz o caminho de volta, com as previsões.
>
> **`ax.contourf(x, y, z, levels, colors, alpha)`** — pinta as regiões da malha pelo valor de `z`: com `levels=[-0.5, 0.5, 1.5]`, o que vale 0 recebe a primeira cor, e o que vale 1, a segunda. **`ax.set_aspect("equal")`** — a mesma escala nos dois eixos, para a diagonal sair a 45°. **`ax.set_xticks`** — as posições das marcas do eixo horizontal, como o `set_yticks` no vertical. **`plt.subplots(2, 2, figsize, sharex, sharey)`** — quatro painéis em duas linhas, com os mesmos eixos.

Na reta, a logística tem a forma certa: a fronteira dela quase coincide com a tracejada, e o erro de teste, 0,1226, fica a 0,0235 do erro do classificador de Bayes nestes pontos, 0,0991. A árvore, com 14 folhas, só consegue imitar a diagonal com uma escada, e cada degrau deixa de um lado ou do outro uma fatia do plano com a classe errada; com 200 pontos, a escada é grosseira, e o erro de teste sobe a 0,2055. No L, os papéis se invertem: a logística não tem como dobrar a reta e corta o L em diagonal, com erro de 0,2011, enquanto a árvore acha os dois cortes com 3 folhas e erra 0,1043, a 0,0052 do erro do classificador de Bayes.

Na reta, a árvore escolhida tem erro de validação cruzada de 0,1300 e erro de teste de 0,2055. Pela validação cruzada, ela quase empataria com a logística, que tem 0,1250; pelo teste, erra 0,0829 a mais. De onde vem a distância entre 0,1300 e 0,2055? O chunk abaixo mede o erro de validação cruzada e o de teste de cada árvore do caminho de poda, e não só da escolhida:

In [ ]:
busca = buscas["reta"]
y_s, y_teste = rotulos["reta"]
alfas = np.array(busca.cv_results_["param_ccp_alpha"], dtype=float)
cv_caminho = 1 - busca.cv_results_["mean_test_score"]
teste_caminho = np.array([
    1 - DecisionTreeClassifier(ccp_alpha=alfa, random_state=12)
    .fit(X, y_s).score(X_teste, y_teste) for alfa in alfas])
print("árvores no caminho:", len(alfas))
print("CV abaixo do teste em todas:",
      bool((cv_caminho < teste_caminho).all()))
sem_poda = alfas == 0
print("sem poda: CV", numero_br(cv_caminho[sem_poda][0], 4),
      "teste", numero_br(teste_caminho[sem_poda][0], 4))

> **🔧 Função**
>
> **`busca.cv_results_["param_ccp_alpha"]`** — os `ccp_alpha` da grade, na ordem em que a busca os testou, alinhados com `mean_test_score`, a acurácia média de validação cruzada de cada um.

Nesta amostra, a validação cruzada fica abaixo do teste em todas as árvores do caminho, inclusive na árvore sem poda, que não foi escolhida por critério nenhum: 0,1600 contra 0,2163, uma distância de 0,0563. Na árvore escolhida, a distância entre 0,1300 e 0,2055 é de 0,0755. As duas distâncias são de árvores diferentes e não se somam nem se subtraem, mas a da árvore sem poda, que nenhuma escolha tocou, indica de onde deve vir boa parte da outra: do sorteio dos 200 pontos de treino. Com esses 200 pontos, a validação cruzada subestima o erro de todas as árvores do caminho, escolhida ou não, e com outro sorteio a distância poderia ser menor ou ir para o outro lado.

A outra causa é o otimismo do mínimo, visto na seção 12.2: entre todas as árvores do caminho, a escolhida é a de menor erro de validação cruzada, e esse erro, 0,1300, fica 0,0300 abaixo dos 0,1600 da árvore sem poda. Parte dessa diferença é sorte da escolha, e o erro do escolhido tende a sair menor do que ele de fato é. Da primeira causa não há como fugir com 200 pontos; a segunda se remove medindo o erro em pontos que não participaram da escolha do `ccp_alpha`.

### Em dados reais

Num conjunto de dados real, não há 10.000 pontos de teste guardados, e a comparação se faz por validação cruzada. Uma árvore podada tem um passo que aprende com o dado, a escolha do `ccp_alpha`, e esse passo precisa ser refeito dentro de cada grupo: como na seção 10.6, toda etapa que aprende do dado entra no `cross_val_score`, e como na seção 11.2, a busca é repetida no treino de cada grupo. A forma direta de fazer isso é dar ao `cross_val_score` um `GridSearchCV` como estimador.

> **🔷 Conceito**
>
> Na **validação cruzada aninhada**, há duas divisões em grupos, uma dentro da outra. Cada grupo **externo** fica de fora uma vez. Com as linhas dos outros grupos externos, a busca roda a sua própria validação cruzada, em grupos **internos**, escolhe o `ccp_alpha` e reajusta a árvore. A árvore escolhida é então medida no grupo externo que ficou de fora, que não participou da escolha. A média nos grupos externos estima o erro de teste do procedimento inteiro, escolha incluída.

O caminho de poda muda com os dados de cada grupo externo, e a grade de um ponto por intervalo teria de ser recalculada em cada um. Uma grade fixa, larga o bastante para ir da árvore inteira à raiz sozinha, evita isso: zero e 41 valores de $10^{-4}$ a 1, em ordem decrescente.

Nos jogadores de `Hitters`, com as 19 colunas e o logaritmo do salário, a árvore podada disputa com a regressão linear, em 10 grupos externos e 5 internos:

In [ ]:
X_h = pd.get_dummies(hitters.drop(columns="salario"), drop_first=True,
                     dtype=float)
grade = np.r_[0.0, np.logspace(-4, 0, 41)][::-1]
externos = KFold(10, shuffle=True, random_state=12)
internos = KFold(5, shuffle=True, random_state=12)
podada = GridSearchCV(DecisionTreeRegressor(random_state=12),
                      {"ccp_alpha": grade}, cv=internos,
                      scoring="neg_mean_squared_error")
mse = {nome: -cross_val_score(modelo, X_h, y, cv=externos,
                              scoring="neg_mean_squared_error")
       for nome, modelo in [("árvore", podada),
                            ("linear", LinearRegression())]}
print("colunas:", X_h.shape[1])
raiz = DecisionTreeRegressor(ccp_alpha=grade[0], random_state=12)
print("folhas com ccp_alpha = 1:", raiz.fit(X_h, y).get_n_leaves())
for nome, valores in mse.items():
    print(f"MSE {nome}: {numero_br(valores.mean(), 4)}")
print("árvore menor em",
      (mse["árvore"] < mse["linear"]).sum(), "de 10 grupos")

> **🔧 Função**
>
> **`np.logspace(a, b, n)`** — `n` números igualmente espaçados em escala logarítmica, de $10^a$ a $10^b$. **`cross_val_score(..., scoring)`** — `scoring="neg_mean_squared_error"` mede o MSE com o sinal trocado, como no `GridSearchCV`. Com uma busca como estimador, cada ajuste num grupo externo é uma busca inteira: validação cruzada interna, escolha do `ccp_alpha` e reajuste com o melhor.

Com `ccp_alpha` = 1, a árvore fica com a raiz sozinha, e com zero, não é podada: a grade cobre os dois extremos. Nos jogadores, a árvore podada tem MSE de validação cruzada aninhada de 0,2323, contra 0,4242 da regressão linear com as 19 colunas, e erra menos em cada um dos 10 grupos externos. Aqui, as caixas se aproximaram mais da relação entre as colunas e o logaritmo do salário do que a soma de efeitos. A regressão linear aqui vai sem penalização; a ridge e o lasso do capítulo 11, que encolhem os coeficientes, seriam outros concorrentes, e a comparação com eles pede a mesma validação cruzada aninhada, com o $\lambda$ escolhido dentro de cada grupo externo.

Nos pacientes de `Heart`, a disputa é entre a árvore de classificação e a regressão logística. A árvore usa as indicadoras de todas as categorias, como na seção 12.3; a logística, com intercepto, descarta a primeira categoria de cada coluna de texto, porque sem isso as indicadoras de uma coluna somam 1 em toda linha e repetem o intercepto:

In [ ]:
coracao = pd.read_csv("dados/Heart.csv").dropna()
y_c = coracao["doenca_cardiaca"]
medidas = coracao.drop(columns="doenca_cardiaca")
X_arvore = pd.get_dummies(medidas, dtype=float)
X_logistica = pd.get_dummies(medidas, drop_first=True, dtype=float)
externos = StratifiedKFold(10, shuffle=True, random_state=12)
internos = StratifiedKFold(5, shuffle=True, random_state=12)
podada = GridSearchCV(DecisionTreeClassifier(random_state=12),
                      {"ccp_alpha": grade}, cv=internos, scoring="accuracy")
logistica = LogisticRegression(C=np.inf, solver="newton-cholesky", tol=1e-8)
erro = {"árvore": 1 - cross_val_score(podada, X_arvore, y_c, cv=externos),
        "logística": 1 - cross_val_score(logistica, X_logistica, y_c,
                                         cv=externos)}
print("colunas:", X_arvore.shape[1], "e", X_logistica.shape[1])
raiz = DecisionTreeClassifier(ccp_alpha=grade[0], random_state=12)
print("folhas com ccp_alpha = 1:",
      raiz.fit(X_arvore, y_c).get_n_leaves())
for nome, valores in erro.items():
    print(f"erro {nome}: {numero_br(valores.mean(), 4)}")
empate = np.isclose(erro["árvore"], erro["logística"])
menor = erro["logística"] < erro["árvore"]
print("logística menor em", (menor & ~empate).sum(), "de 10")
print("empate em", empate.sum(), "de 10")
print("árvore menor em", (~menor & ~empate).sum(), "de 10")

As 18 colunas da árvore viram 16 na logística, uma a menos por coluna de texto. Nos pacientes, a logística tem o menor erro de validação cruzada aninhada, 0,1683 contra 0,2186 da árvore podada, e erra menos em 7 dos 10 grupos externos, empata em 2 e erra mais em 1. Cada método ganha num dos dois conjuntos: nos jogadores, as caixas de (8.9); nos pacientes, a logística, que é linear no logaritmo da chance, como a regressão de (8.8) é linear na resposta.

O erro de teste também não é o único critério. A árvore podada da seção 12.3, com perguntas sobre o exame de tálio, a dor no peito e os vasos coloridos, cabe numa folha de papel e se explica a quem nunca ouviu falar de uma logística. Quanto de erro vale a pena pagar por essa clareza?

## Vantagens e Desvantagens das Árvores

> **📌 Nota**
>
> Esta seção corresponde à seção 8.1.4 de James et al. (2023).

O dirigente de um clube de beisebol quer saber por que o modelo prevê um salário alto para um jogador. Uma árvore responde com as perguntas que o jogador percorreu até a folha: quantas vezes foi ao bastão na carreira, quantas rebatidas fez na temporada. Uma regressão linear responde com uma soma de parcelas, uma por coluna, e cada parcela só faz sentido com as outras colunas fixas. Essa clareza tem preço, e o preço aparece quando a amostra de treino muda.

In [ ]:
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     train_test_split)
from sklearn.tree import (DecisionTreeClassifier, DecisionTreeRegressor,
                          plot_tree)

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### O que as árvores têm a favor

A primeira vantagem já apareceu nas figuras das seções anteriores: uma árvore pequena se desenha numa folha de papel, e quem a lê não precisa saber o que é um coeficiente, uma média ponderada ou um logaritmo da chance. Basta seguir as perguntas de cima para baixo. Explicar uma regressão linear exige mais: dizer que cada coeficiente é o efeito de uma coluna com as outras fixas, e que o efeito é o mesmo para qualquer valor delas. A clareza vale enquanto a árvore é pequena. A árvore sem poda da seção 12.2 também é uma sequência de perguntas, mas tem folhas demais para ser lida de cima para baixo.

A segunda vantagem, a mais discutível, é a semelhança com o modo como as pessoas decidem: um médico que primeiro olha o exame de tálio e, conforme o resultado, pergunta pelo tipo da dor no peito segue um roteiro com a forma de uma árvore. É uma impressão, difícil de medir, e as regras que uma pessoa usa não precisam ser as que a árvore acha nos dados.

A terceira vantagem é de método. Uma coluna de texto com $K$ categorias pode ser dividida sem virar números: o corte manda um grupo de categorias para um lado e o resto para o outro. Com as $K$ categorias em dois grupos não vazios, sem importar qual grupo vai para a esquerda, há $2^{K-1} - 1$ divisões possíveis. A coluna `dor_no_peito` dos pacientes de `Heart` é um exemplo:

In [ ]:
coracao = pd.read_csv("dados/Heart.csv").dropna()
tipos = sorted(coracao["dor_no_peito"].unique())
K = len(tipos)
print("categorias:", K)
print("2^(K−1) − 1 =", 2 ** (K - 1) - 1)
divisoes = [set(grupo) for tamanho in range(1, K)
            for grupo in combinations(tipos, tamanho)
            if tipos[0] in grupo]
print("divisões listadas:", len(divisoes))
curto = {"assintomática": "ass", "atípica": "atí",
         "não anginosa": "não", "típica": "típ"}
for d in divisoes:
    fora = [t for t in tipos if t not in d]
    print(" ", " ".join(curto[t] for t in sorted(d)), "|",
          " ".join(curto[t] for t in fora))
sozinha = [d for d in divisoes if len(d) in (1, K - 1)]
print("uma categoria contra as outras:", len(sozinha))
print("duas contra duas:", len(divisoes) - len(sozinha))

> **🔧 Função**
>
> **`combinations(itens, tamanho)`**, do módulo `itertools` — todos os grupos de `tamanho` itens, sem repetir e sem importar a ordem. Um grupo e o seu complemento são a mesma divisão, por isso a lista fica só com os grupos que contêm a primeira categoria.

Na lista, cada categoria aparece pelas três primeiras letras (`ass` é a assintomática, `atí` a atípica, `não` a não anginosa e `típ` a típica), e a barra separa os dois lados do corte. As 4 categorias dão 7 divisões: em 4 delas, uma categoria fica sozinha contra as outras três; nas outras 3, as categorias vão duas para cada lado. O `DecisionTreeClassifier` não recebe texto, e as colunas de texto entram como indicadoras, uma por categoria. Um corte numa indicadora, como `dor_no_peito_assintomática ≤ 0,5`, separa uma categoria das demais. Das 7 divisões, só as 4 de uma categoria contra as outras três cabem num corte; as 3 de duas contra duas, não. Com o `scikit-learn`, portanto, a vantagem vale só em parte.

O chunk abaixo refaz a escolha da seção 12.3, com a mesma grade de `ccp_alpha` e a mesma validação cruzada, chega à árvore de 6 folhas e lista os cortes dela em indicadoras de `dor_no_peito`, com o ramo da raiz em que cada um está:

In [ ]:
y_c = coracao["doenca_cardiaca"]
X_c = pd.get_dummies(coracao.drop(columns="doenca_cardiaca"), dtype=float)
grande = DecisionTreeClassifier(random_state=12).fit(X_c, y_c)
a = grande.cost_complexity_pruning_path(X_c, y_c).ccp_alphas
grade = np.r_[0.0, np.sqrt(a[1:-1] * a[2:])]
busca = GridSearchCV(DecisionTreeClassifier(random_state=12),
                     {"ccp_alpha": grade[::-1]},
                     cv=StratifiedKFold(10, shuffle=True, random_state=12),
                     scoring="accuracy").fit(X_c, y_c)
seis = busca.best_estimator_
print("folhas:", seis.get_n_leaves())
no = seis.tree_
print("raiz:", X_c.columns[no.feature[0]], "≤",
      numero_br(no.threshold[0], 1))
for i in np.flatnonzero(no.feature >= 0):
    coluna = X_c.columns[no.feature[i]]
    if coluna.startswith("dor_no_peito"):
        ramo = "sim" if i < no.children_right[0] else "não"
        categoria = coluna.removeprefix("dor_no_peito_")
        print(f"  nó {i}: …_{categoria}, ramo {ramo}")

> **🔧 Função**
>
> **`arvore.tree_.children_right`** — para cada nó, o número do filho do ramo "não", o da direita. A árvore numera os nós a partir da raiz, descendo primeiro pelo ramo "sim" até o fim; por isso os nós de número menor que `children_right[0]` são os do ramo "sim" da raiz, e os outros, os do ramo "não".

A raiz pergunta pela indicadora do tálio normal, e o ramo "sim" é o dos pacientes cujo exame de tálio não deu normal. Os dois cortes em `dor_no_peito` são na mesma indicadora, a da dor assintomática, um em cada ramo da raiz: entre os pacientes com tálio normal e entre os outros, a árvore separa essa categoria das outras três. Uma árvore que recebesse a coluna de texto inteira poderia também, num corte só, mandar a dor típica e a atípica para um lado e a assintomática e a não anginosa para o outro. Com indicadoras, esse corte não existe: dois cortes seguidos deixam sempre três grupos e quebram um dos dois pares. Cortando na típica e depois na atípica, as duas ficam separadas uma da outra; cortando na assintomática e depois na não anginosa, são essas duas que se separam.

### O que as árvores têm contra

A primeira desvantagem é a precisão. Uma árvore prevê bem quando a relação entre os preditores e a resposta se parece com caixas, e em muitos problemas outros métodos de regressão e de classificação erram menos. Na seção 12.4, a árvore podada venceu a regressão linear sem penalização nos jogadores de `Hitters` e a regressão logística no simulado de fronteira em L, mas perdeu para a logística nos pacientes de `Heart` e no simulado de fronteira diagonal, que ela só imitava em escada.

A segunda é a **instabilidade**: a árvore ajustada depende muito da amostra de treino, e duas amostras do mesmo grupo de jogadores podem dar árvores bem diferentes. O chunk abaixo divide os 263 jogadores de `Hitters` em duas metades sorteadas, a metade A com 132 jogadores e a B com os 131 restantes, e cresce em cada uma uma árvore de 6 folhas, com as mesmas 19 colunas e a mesma semente. Antes de rodá-lo: as duas metades saem do mesmo grupo de jogadores e passam pela mesma receita. Quantas das colunas usadas pelas duas árvores você espera que sejam as mesmas?

In [ ]:
hitters = pd.read_csv("dados/Hitters.csv").dropna()
y = np.log(hitters["salario"])
X = pd.get_dummies(hitters.drop(columns="salario"), drop_first=True,
                   dtype=float)
A_X, B_X, A_y, B_y = train_test_split(X, y, train_size=132,
                                      random_state=12)
print("jogadores:", len(X), "  colunas:", X.shape[1])
arvores, usadas, perguntas = {}, {}, {}
for nome, X_m, y_m in [("A", A_X, A_y), ("B", B_X, B_y)]:
    arvore = DecisionTreeRegressor(max_leaf_nodes=6, random_state=12)
    arvores[nome] = arvore.fit(X_m, y_m)
    no = arvore.tree_
    print(f"metade {nome}: {len(y_m)} jogadores, raiz em")
    print(" ", X.columns[no.feature[0]], "≤",
          numero_br(no.threshold[0], 1))
    internos = no.feature >= 0
    usadas[nome] = set(X.columns[no.feature[internos]])
    perguntas[nome] = set(zip(X.columns[no.feature[internos]],
                              no.threshold[internos]))
    print("  colunas usadas:", len(usadas[nome]))
print("usadas pelas duas:")
for coluna in sorted(usadas["A"] & usadas["B"]):
    print(" ", coluna)
print("coluna e limiar iguais:",
      len(perguntas["A"] & perguntas["B"]))

A metade A é o conjunto de treino da seção 12.2, e a árvore dela começa por `vezes_ao_bastao_carreira`, com corte em 1.452,0. A da metade B começa por `corridas_carreira`, com corte em 204,5. A árvore A usa 5 colunas, a B usa 4, e só `impulsionadas_carreira` aparece nas duas, com limiares diferentes: nenhum par de coluna e limiar se repete. As colunas `impulsionadas` e `rebatidas`, sem o sufixo, contam só a temporada, e são colunas diferentes de `impulsionadas_carreira` e `rebatidas_carreira`.

Para desenhar cada árvore de seis folhas na largura da página, a função `em_portugues` reescreve o texto das caixas, como na seção 12.3. Nos nós internos ficam a pergunta, com o limiar na linha de baixo (o `split(" <= ")` separa a coluna do limiar), e o número de jogadores; nas folhas, o número de jogadores e a média do logaritmo do salário. Os números saem com a pontuação brasileira, pela `numero_br`, e os rótulos "sim" e "não" são afastados da caixa da raiz, para o lado de fora de cada ramo.

In [ ]:
def em_portugues(anotacoes):
    """Traduz e ajusta as caixas que o plot_tree desenha."""
    for anotacao in anotacoes:
        linhas = (anotacao.get_text().replace("samples = ", "n=")
                  .replace("True", "sim").replace("False", "não")
                  .split("\n"))
        if " <= " in linhas[0]:  # nó interno
            coluna, limiar = linhas[0].split(" <= ")
            coluna = coluna.replace("_carreira", "_\ncarreira")
            limiar = numero_br(float(limiar), 1)
            texto = f"{coluna}\n≤ {limiar}\n{linhas[1]}"
        elif len(linhas) > 1:  # folha
            media = float(linhas[1].replace("value = ", ""))
            texto = f"{linhas[0]}\n{numero_br(media, 2)}"
        else:  # rótulo "sim" ou "não", afastado para fora do ramo
            texto = linhas[0].strip()
            x, y_texto = anotacao.get_position()
            lado = -1 if texto == "sim" else 1
            anotacao.set_position((x + lado * 0.07, y_texto))
        anotacao.set_text(texto)
        caixa = anotacao.get_bbox_patch()
        if caixa is not None:
            caixa.set(facecolor="white", edgecolor="0.45",
                      boxstyle="round,pad=0.2")

> **🔧 Função**
>
> **`anotacao.get_position()`** e **`anotacao.set_position((x, y))`** — leem e mudam a posição de um texto da figura; nas caixas do `plot_tree`, em fração do painel, de 0 a 1. Na `em_portugues`, levam o "sim" 7% da largura do painel para a esquerda e o "não" 7% para a direita, longe da caixa da raiz.

In [ ]:
# Figura: A árvore de 6 folhas ajustada na metade A, com 132 jogadores, para o logaritmo do salário. Nos nós internos, a pergunta e o número de jogadores (n); nas folhas, o número de jogadores e, embaixo, a média do logaritmo do salário deles. O ramo "sim", à esquerda, é o da condição verdadeira.
fig, ax = plt.subplots(figsize=(4.4, 4.4))
caixas = plot_tree(arvores["A"], feature_names=list(X.columns),
                   impurity=False, rounded=True, precision=2,
                   fontsize=9, ax=ax)
em_portugues(caixas)
ax.set_title("metade A")
plt.show()

A árvore da metade B, desenhada da mesma forma:

In [ ]:
# Figura: A árvore de 6 folhas ajustada na metade B, com os outros 131 jogadores, as mesmas 19 colunas e a mesma semente. O texto das caixas segue a figura da metade A.
fig, ax = plt.subplots(figsize=(4.2, 4.4))
caixas = plot_tree(arvores["B"], feature_names=list(X.columns),
                   impurity=False, rounded=True, precision=2,
                   fontsize=9, ax=ax)
em_portugues(caixas)
ax.set_title("metade B")
plt.show()

As duas árvores começam por colunas diferentes e seguem por caminhos diferentes. De onde vem tanta diferença, se as duas metades saem do mesmo grupo de jogadores? A raiz é o corte que mais reduz o RSS, e o chunk abaixo mede, em cada metade, a fração do RSS da árvore sem cortes, a que prevê a média para todos, que o melhor corte de cada coluna, sozinho, remove. Na tabela, a coluna A traz as frações da metade A, e a coluna B, as da metade B:

In [ ]:
def fracao_removida(X_m, y_m):
    """Fração do RSS da média que o melhor corte de cada coluna remove."""
    tss = ((y_m - y_m.mean()) ** 2).sum()
    fracao = {}
    for coluna in X_m.columns:
        toco = DecisionTreeRegressor(max_depth=1, random_state=12)
        previsto = toco.fit(X_m[[coluna]], y_m).predict(X_m[[coluna]])
        fracao[coluna] = 1 - ((y_m - previsto) ** 2).sum() / tss
    return pd.Series(fracao)

cortes = pd.DataFrame({"A": fracao_removida(A_X, A_y),
                       "B": fracao_removida(B_X, B_y)})
for nome in ["A", "B"]:
    print(nome, "melhor é a raiz:", cortes[nome].idxmax()
          == X.columns[arvores[nome].tree_.feature[0]])
cinco = {m: set(cortes[m].nlargest(5).index) for m in ["A", "B"]}
print("5 primeiras iguais nas duas:", cinco["A"] == cinco["B"])
(cortes.loc[sorted(cinco["A"])].sort_values("A", ascending=False)
 .style.format(lambda v: numero_br(v, 3)))

> **🔧 Função**
>
> **`DecisionTreeRegressor(max_depth)`** — `max_depth=1` limita a árvore a uma pergunta só, a raiz e duas folhas; ajustada com uma coluna, faz o melhor corte naquela coluna. **`s.nlargest(5)`** — os cinco maiores valores da série. **`df.sort_values(coluna, ascending=False)`** — as linhas em ordem decrescente da coluna.

Nas duas metades, a raiz da árvore é a coluna do melhor corte isolado, e as cinco colunas com os melhores cortes são as mesmas: cinco contagens acumuladas na carreira. No topo da tabela, a disputa é apertada. Na metade B, o corte em `corridas_carreira` remove 0,556 do RSS e o em `vezes_ao_bastao_carreira`, 0,554, uma diferença de 0,002; na A, a ordem se inverte: 0,578 para `corridas_carreira` e 0,602 para `vezes_ao_bastao_carreira`. As cinco colunas medem quase a mesma coisa, o tempo e o volume de jogo de cada jogador:

In [ ]:
correlacoes = hitters[sorted(cinco["A"])].corr()
pares = correlacoes.to_numpy()[np.triu_indices(5, 1)]
print("pares de colunas:", len(pares))
print("menor correlação:", numero_br(pares.min(), 3))
print("maior correlação:", numero_br(pares.max(), 3))

> **🔧 Função**
>
> **`df.corr()`** — a matriz de correlações entre todas as colunas numéricas da tabela, uma linha e uma coluna para cada. **`np.triu_indices(n, 1)`** — as posições acima da diagonal de uma matriz $n \times n$; aqui, cada par de colunas uma vez só.

As correlações entre as cinco, nos 10 pares, vão de 0,889 a 0,995. Com colunas quase repetidas, a coluna escrita na raiz pode trocar de uma amostra para outra sem que a divisão dos jogadores mude muito. O chunk abaixo aplica as duas perguntas da raiz aos 263 jogadores e conta quantos elas mandam para o mesmo lado:

In [ ]:
lado = {}
for nome in ["A", "B"]:
    no = arvores[nome].tree_
    coluna = X.columns[no.feature[0]]
    lado[nome] = X[coluna] <= no.threshold[0]
iguais = (lado["A"] == lado["B"]).sum()
print("mesmo lado:", iguais, "de", len(X))
print("fração:", numero_br(iguais / len(X), 3))

As duas perguntas mandam 253 dos 263 jogadores para o mesmo lado: a raiz trocou de coluna, mas separou quase os mesmos jogadores. A diferença entre as árvores está abaixo da raiz. O corte que vence em cada nó depende só dos jogadores que chegam a ele, e as metades A e B não têm nenhum jogador em comum. E a divisão gulosa da seção 12.1 decide cada nó sem voltar atrás.

Estrutura diferente, sozinha, não quer dizer previsão diferente: duas árvores com perguntas diferentes podem prever quase o mesmo salário para cada jogador. A variância da seção 7.6 é o quanto a previsão $\hat f(x_0)$ muda de um conjunto de treino para outro. O chunk abaixo a mede: sorteia 200 amostras de 132 jogadores, ajusta em cada uma a árvore de 6 folhas e uma regressão linear sem penalização, com as mesmas 19 colunas, e prevê os 263 jogadores com cada modelo. Para cada jogador, calcula a variância das 200 previsões e no fim tira a média entre os jogadores:

In [ ]:
previsoes = {"árvore": [], "linear": []}
for semente in range(200):
    X_t, _, y_t, _ = train_test_split(X, y, train_size=132,
                                      random_state=semente)
    arvore = DecisionTreeRegressor(max_leaf_nodes=6, random_state=12)
    previsoes["árvore"].append(arvore.fit(X_t, y_t).predict(X))
    linear = LinearRegression().fit(X_t, y_t)
    previsoes["linear"].append(linear.predict(X))
variancia = {modelo: np.var(np.array(p), axis=0).mean()
             for modelo, p in previsoes.items()}
for modelo, v in variancia.items():
    print(f"{modelo}: {numero_br(v, 3)}")
print("árvore varia mais:", variancia["árvore"] > variancia["linear"])

> **🔧 Função**
>
> **`np.var(matriz, axis=0)`** — a variância de cada coluna da matriz. Aqui, cada linha são as previsões de uma amostra e cada coluna é um jogador, e `.mean()` tira a média das 263 variâncias.

A previsão da árvore para um mesmo jogador tem variância média de 0,077 entre as 200 amostras, e a da regressão linear, de 0,041. Neste problema, a árvore tem variância maior que a da regressão linear: ela muda de estrutura e também de previsão quando a amostra muda. Para ver o que essa média resume, o chunk abaixo escolhe um jogador só, o primeiro da tabela entre os de salário igual à mediana, e separa as 200 previsões de cada modelo para ele:

In [ ]:
mediano = (hitters["salario"] - hitters["salario"].median()).abs().idxmin()
j = hitters.index.get_loc(mediano)
print("salário mediano:", numero_br(hitters["salario"].median(), 1))
print("salário do jogador:", numero_br(hitters.loc[mediano, "salario"], 1))
print("jogadores com esse salário:",
      (hitters["salario"] == hitters.loc[mediano, "salario"]).sum())
print("log do salário:", numero_br(y.iloc[j]))
do_jogador = {m: np.array(p)[:, j] for m, p in previsoes.items()}
for modelo, prev in do_jogador.items():
    print(f"{modelo}: de {numero_br(prev.min())} a {numero_br(prev.max())}")
    print("  variância:", numero_br(prev.var(), 3))
    print("  abaixo do real:", (prev < y.iloc[j]).sum(), "de 200")

> **🔧 Função**
>
> **`indice.get_loc(rotulo)`** — a posição, contada de zero, da linha que tem aquele rótulo no índice. As previsões guardadas em `previsoes` estão em ordem de linha, e é a posição que as indexa.

In [ ]:
# Figura: As 200 previsões do logaritmo do salário para um jogador de salário igual à mediana, uma por amostra de 132 jogadores. Em cima, a regressão linear; embaixo, a árvore de 6 folhas. A linha tracejada é o logaritmo do salário real dele.
fig, eixos = plt.subplots(2, 1, figsize=(4.4, 4.0), sharex=True,
                          sharey=True)
fig.subplots_adjust(hspace=0.45)
todas = np.concatenate(list(do_jogador.values()))
faixas = np.linspace(todas.min(), todas.max(), 31)
for ax, modelo, cor, titulo in [
        (eixos[0], "linear", "C0", "regressão linear"),
        (eixos[1], "árvore", "C1", "árvore de 6 folhas")]:
    ax.hist(do_jogador[modelo], bins=faixas, color=cor,
            edgecolor="none")
    ax.axvline(y.iloc[j], color="0.45", linestyle="--", linewidth=1.2)
    ax.set_title(titulo, loc="left")
    ax.set_ylabel("amostras")
eixos[1].set_xlabel("logaritmo do salário previsto")
plt.show()

> **🔧 Função**
>
> **`ax.hist(valores, bins)`** — o histograma dos valores; `bins` recebe as bordas das faixas, as mesmas nos dois painéis para que as barras se comparem.

O jogador ganha 425,0 mil dólares, a mediana, como outros dois. As previsões da regressão linear para ele vão de 4,98 a 5,80, com variância 0,023; as da árvore vão de 4,61 a 7,06, com variância 0,147. De uma amostra para outra, o jogador cai em folhas diferentes, e cada folha dá um valor, por isso o histograma da árvore sai em grupos separados. A linear erra de outro jeito: as 200 previsões dela ficam abaixo do logaritmo do salário real, 6,05, e só 57 das 200 da árvore ficam.

Variância maior não quer dizer erro maior: o erro de teste esperado soma a variância, o viés ao quadrado e o erro irredutível (seção 7.6), e na seção 12.4 a árvore podada errou menos que a linear nesses jogadores.

Na prática, a árvore é uma boa escolha quando é preciso explicar a previsão a quem não é técnico, ou quando a relação tem limiares e interações, como no simulado em L da seção 12.4. Quando só o erro de teste conta, ela entra na comparação por validação cruzada, como na seção 12.4, e perde quando outro modelo erra menos. Juntar muitas árvores, cada uma ajustada numa amostra diferente dos dados, e combinar as previsões delas ataca justamente a variância; é o assunto do capítulo 13.

## Leituras adicionais

- O capítulo 8 de James et al. (2023). O PDF gratuito e os dados estão no [site oficial do livro](https://www.statlearning.com).
- A seção 9.2 de Hastie et al. (2009), com as árvores de regressão e de classificação, a poda por custo-complexidade e a divisão de preditores categóricos num tratamento mais técnico.
- Breiman (2001), sobre as duas culturas da modelagem estatística. A seção 8, "Rashomon and the multiplicity of good models", relata na p. 206 que uma pequena perturbação do treino pode dar uma árvore bem diferente, com quase o mesmo erro de teste.
- [O guia de árvores de decisão do scikit-learn](https://scikit-learn.org/stable/modules/tree.html), com o `DecisionTreeRegressor`, o `DecisionTreeClassifier`, os critérios de impureza e a poda por custo-complexidade.
- [O exemplo de poda por custo-complexidade do scikit-learn](https://scikit-learn.org/stable/auto_examples/tree/plot_cost_complexity_pruning.html), com o `cost_complexity_pruning_path` e o número de nós, a profundidade e a acurácia de treino e de teste ao longo do `ccp_alpha`.

## Referências

- **Breiman**. *Statistical Modeling: The Two Cultures*. 2001.
- **Breiman**. *Random Forests*. 2001.
- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed. Springer. 2009.
- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.